# cleaning ground truth

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd

# Raw CGWB data

In [ ]:
cgwb=pd.read_csv(f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/input/CGWB_Time_Series_GW - CGWB_Time_Series_GW.csv')

In [ ]:
cgwb

In [ ]:
for col in cgwb.columns:
  print(col)




**1. Selecting only values from 2001**

In [ ]:
columns_to_keep = ['objectid', 'pointInRegion', 'station_id', 'station_name', '.geo'] + \
                  [col for col in cgwb.columns if col.startswith('2001') or (col.startswith('20') and not col.startswith('2000'))]

In [ ]:
cgwb_2001_onwards = cgwb[columns_to_keep]

In [ ]:
# Define the mapping dictionary for column renaming
column_mapping = {
    'objectid': 'objectid',
    'pointInRegion': 'pointInRegion',
    'station_id': 'station_id',
    'station_name': 'station_name',
    '.geo': '.geo'
}

# Rename the columns representing seasons
for year in range(2001, 2024):
    column_mapping[f'{year}_season1'] = f'{year}_Jan_March'
    column_mapping[f'{year}_season2'] = f'{year}_April_June'
    column_mapping[f'{year}_season3'] = f'{year}_Jul_Sept'
    column_mapping[f'{year}_season4'] = f'{year}_Oct_Dec'

# Apply column renaming
cgwb_2001_onwards_renamed = cgwb_2001_onwards.rename(columns=column_mapping)

In [ ]:
cgwb_2001_onwards_renamed

In [ ]:
cgwb_2001_onwards_renamed[cgwb_2001_onwards_renamed['objectid'] == 3181]

In [ ]:
cgwb_2001_onwards_renamed[cgwb_2001_onwards_renamed['station_id'] ==  303310076565901]

**2. Displaying the available wells in CGWB data as points on India map**




In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import json

# Assuming 'cgwb_2001_onwards_renamed' is your DataFrame
# Create a function to convert the .geo column to Shapely Point objects
def convert_geo_to_point(geo_str):
    geo_json = json.loads(geo_str)
    coordinates = geo_json['coordinates']
    return Point(coordinates)

# Create a function to convert the .geo column to Shapely Point objects
def convert_geo_to_point(geo_str):
    geo_json = json.loads(geo_str)
    coordinates = geo_json['coordinates']
    return Point(coordinates)

# Apply the conversion function to create a 'geometry' column
cgwb_2001_onwards_renamed['geometry'] = cgwb_2001_onwards_renamed['.geo'].apply(convert_geo_to_point)

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame(cgwb_2001_onwards_renamed, geometry='geometry')

# Plot the GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, marker='o', color='blue', markersize=0.5, edgecolor='black', label='Well')

# Add title and axes labels
plt.title('Map of Well Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.legend()
plt.show()

In [ ]:
import numpy as np

**3. Finding out the number of consecutive data available per well**





In [ ]:
import pandas as pd

# Sample dataframe structure based on your provided data
# Assuming 'cgwb_2001_onwards_renamed' is your dataframe
# Remove the first four columns for the calculation
data = cgwb_2001_onwards_renamed.iloc[:, 4:]

# Function to calculate consecutive non-NaN pairs excluding the first four columns
def consecutive_non_nan_pairs(row):
    non_nan_pairs = 0
    consecutive_count = 0
    for value in row:
        if not pd.isna(value):
            consecutive_count += 1
            if consecutive_count == 2:  # Check if two consecutive non-NaN values are found
                non_nan_pairs += 1
                consecutive_count = 1  # Reset consecutive count to 1 to check for overlapping pairs
        else:
            consecutive_count = 0  # Reset consecutive count if NaN is encountered
    return non_nan_pairs

# Apply the function to each row in the dataframe
cgwb_2001_onwards_renamed['consecutive_non_nan_pairs'] = data.apply(consecutive_non_nan_pairs, axis=1)

# Create a new dataframe to store the results
results = cgwb_2001_onwards_renamed[['station_id', 'consecutive_non_nan_pairs']]

# Display the results
print(results)




**4. Take only wells above Madhya Pradesh**



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import json

# Assuming 'cgwb_2001_onwards_renamed' is your DataFrame

# Create a function to convert the .geo column to Shapely Point objects
def convert_geo_to_point(geo_str):
    geo_json = json.loads(geo_str)
    coordinates = geo_json['coordinates']
    return Point(coordinates)

# Apply the conversion function to create a 'geometry' column
cgwb_2001_onwards_renamed['geometry'] = cgwb_2001_onwards_renamed['.geo'].apply(convert_geo_to_point)

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame(cgwb_2001_onwards_renamed, geometry='geometry')

# Filter the GeoDataFrame to include only wells within the specified coordinates
filtered_above_MP_gdf = gdf[(gdf.geometry.y >= 21) & (gdf.geometry.y <= 56) &
                   (gdf.geometry.x >= 30) & (gdf.geometry.x <= 100)]

# Plot the filtered GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
filtered_above_MP_gdf.plot(ax=ax, marker='o', color='blue', markersize=2, edgecolor='black', label='Well')

# Add title and axes labels
plt.title('Map of Well Locations within Specified Coordinates (MP and above)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.legend()
plt.show()

In [ ]:
filtered_above_MP_gdf

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Assuming 'cgwb_2001_onwards_renamed' is your dataframe
# The results dataframe already contains the necessary data
results = filtered_above_MP_gdf[['station_id', 'consecutive_non_nan_pairs']]

# Calculate mean, median, and mode
mean_value = results['consecutive_non_nan_pairs'].mean()
median_value = results['consecutive_non_nan_pairs'].median()

print(f"Mean: {mean_value}")
print(f"Median: {median_value}")

# Calculate quartiles and percentiles
Q1 = results['consecutive_non_nan_pairs'].quantile(0.25)
Q3 = results['consecutive_non_nan_pairs'].quantile(0.75)
P10 = results['consecutive_non_nan_pairs'].quantile(0.10)
P20 = results['consecutive_non_nan_pairs'].quantile(0.20)

# Calculate the number of wells for each percentile and quartile
num_wells_P10 = results[results['consecutive_non_nan_pairs'] <= P10].shape[0]
num_wells_P20 = results[results['consecutive_non_nan_pairs'] <= P20].shape[0]
num_wells_Q1 = results[results['consecutive_non_nan_pairs'] <= Q1].shape[0]
num_wells_median = results[results['consecutive_non_nan_pairs'] <= median_value].shape[0]
num_wells_Q3 = results[results['consecutive_non_nan_pairs'] <= Q3].shape[0]

# Total number of wells
total_wells = results.shape[0]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(results['consecutive_non_nan_pairs'], bins=range(0, max(results['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
plt.title('Histogram of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.xlabel('Number of consecutive Non zero pair of values')
plt.ylabel('Number of Wells')
plt.xticks(range(0, max(results['consecutive_non_nan_pairs']) + 5, 5))
plt.grid(axis='y')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                         horizontalalignment='right', fontsize=10)
plt.show()

# Plot inverted box plot
plt.figure(figsize=(10, 6))
sns.boxplot(y=results['consecutive_non_nan_pairs'], color='lightyellow')
plt.title('Box Plot of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.ylabel('Number of consecutive Non zero pair of values')

# Annotate the quartiles and percentiles
plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

# Remove x-axis label as it's not used in the plot
plt.xlabel('')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                         f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                         horizontalalignment='right', fontsize=10)
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Loading Aquifer Data

In [ ]:
import pickle

In [ ]:
# Path to the pickle file
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/aquifer_gdf.pickle'

# Load the GeoDataFrame from the pickle file
with open(pickle_path, 'rb') as f:
    loaded_aquifer_gdf = pickle.load(f)

# Verify the loaded GeoDataFrame
print(loaded_aquifer_gdf.head())

In [ ]:
# Plot the data if available
fig, ax = plt.subplots(figsize=(10, 7))

# Plot Aquifer Data if available
if not loaded_aquifer_gdf.empty:
    loaded_aquifer_gdf.plot(ax=ax, color='blue', edgecolor='black')
    ax.set_title('Aquifer Data')
else:
    ax.set_title('Aquifer Data (Not Available)')

plt.tight_layout()
plt.show()

In [ ]:
loaded_aquifer_gdf.columns

In [ ]:
loaded_aquifer_gdf.rename(columns={'id':'aquifer_id','geometry': 'aquifer_geometry'})

In [ ]:
import pandas as pd

# Assuming 'aquifer_gdf' is your dataframe
# List unique values in 'aquifer' column
unique_aquifer = loaded_aquifer_gdf['aquifer'].unique()
print(f"Unique values in 'aquifer' column:\n{unique_aquifer}\n")

# List unique values in 'aquifer0' column
unique_aquifer0 = loaded_aquifer_gdf['aquifer0'].unique()
print(f"Unique values in 'aquifer0' column:\n{unique_aquifer0}\n")

# List unique values in 'aquifers' column
unique_aquifers = loaded_aquifer_gdf['aquifers'].unique()
print(f"Unique values in 'aquifers' column:\n{unique_aquifers}\n")

In [ ]:
unique_states = loaded_aquifer_gdf['state'].unique()
print(f"Unique states in state columns:\n{unique_states}\n")

In [ ]:
import pandas as pd

# Assuming 'aquifer_gdf' is your dataframe
# Total number of rows in the dataframe
total_rows = loaded_aquifer_gdf.shape[0]
print(f"Total number of rows: {total_rows}")

# Number of rows per 'aquifer'
rows_per_aquifer = loaded_aquifer_gdf['aquifer'].value_counts()
print("\nNumber of rows per 'aquifer':")
print(rows_per_aquifer)

# Number of rows per 'aquifer0' (sub-classification of 'aquifer')
rows_per_aquifer0 = loaded_aquifer_gdf.groupby(['aquifer', 'aquifer0']).size()
print("\nNumber of rows per sub-classification 'aquifer0' within each 'aquifer':")
print(rows_per_aquifer0)

In [ ]:
# Create a dictionary to store aquifers and their subtypes
aquifer_dict = {}

# Group by 'aquifer' and aggregate the unique subtypes in 'aquifer0'
grouped = loaded_aquifer_gdf.groupby('aquifer')['aquifer0'].unique()

# Convert to dictionary
aquifer_dict = grouped.to_dict()

# Display the dictionary
for aquifer, subtypes in aquifer_dict.items():
    print(f"{aquifer}: {list(subtypes)}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt


# Create a color map with unique colors for each aquifer type
unique_aquifers = loaded_aquifer_gdf['aquifer'].unique()
color_map = plt.cm.get_cmap('tab20', len(unique_aquifers))

# Create a dictionary to map each aquifer type to a color
aquifer_colors = {aquifer: color_map(i) for i, aquifer in enumerate(unique_aquifers)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for aquifer, color in aquifer_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer'] == aquifer]
    subset.plot(ax=ax, color=color, label=aquifer)

# Add a legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='Aquifer Types')

# Set title and axes labels
plt.title('Aquifer Map')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# List of states to filter
#states_to_display = ['AN', 'DD', 'GA', 'LK', 'LD', 'PY', 'MS', 'ADJACENT COUNTRIES', 'CH', 'DISPUTED', 'CG', 'MN', 'TG', 'DL']
states_to_display = ['ADJACENT COUNTRIES', 'DISPUTED']

# Filter the GeoDataFrame for the specified states
filtered_aquifer_gdf = loaded_aquifer_gdf[loaded_aquifer_gdf['state'].isin(states_to_display)]

# Display the filtered data
print(filtered_aquifer_gdf)

# Create a color map with unique colors for each state
unique_states = filtered_aquifer_gdf['state'].unique()
color_map = plt.cm.get_cmap('tab20', len(unique_states))

# Create a dictionary to map each state to a hex color
state_colors = {state: mcolors.to_hex(color_map(i)) for i, state in enumerate(unique_states)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for state, color in state_colors.items():
    subset = filtered_aquifer_gdf[filtered_aquifer_gdf['state'] == state]
    subset.plot(ax=ax, color=color, edgecolor='black', label=state)

# Add a legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='States')

# Set title and axes labels
plt.title('Aquifer Map for Selected States')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# List of states to filter
states_to_display = ['AR', 'AS', 'MN', 'ML', 'MZ', 'NL', 'TR', 'SK','MS','UK','LK','HP','JK']
# Filter the GeoDataFrame for the specified states
filtered_aquifer_gdf = loaded_aquifer_gdf[loaded_aquifer_gdf['state'].isin(states_to_display)]

# Display the filtered data
print(filtered_aquifer_gdf)

# Create a color map with unique colors for each state
unique_states = filtered_aquifer_gdf['state'].unique()
color_map = plt.cm.get_cmap('tab20', len(unique_states))

# Create a dictionary to map each state to a hex color
state_colors = {state: mcolors.to_hex(color_map(i)) for i, state in enumerate(unique_states)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for state, color in state_colors.items():
    subset = filtered_aquifer_gdf[filtered_aquifer_gdf['state'] == state]
    subset.plot(ax=ax, color=color, edgecolor='black', label=state)

# Add a legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='States')

# Set title and axes labels
plt.title('Aquifer Map for Selected States')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

# Creating a condensed aquifer list from ACWADAM map

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
# Define the categories and the corresponding aquifers and their subtypes
aquifer_categories = {
    'Crystalline Systems': ['Basalt', 'Granite', 'Schist', 'Quartzite', 'Charnockite', 'Basement Gneissic Complex', 'Gneiss', 'Intrusive', 'Khondalites'],
    'Sedimentary Hard Rock': ['Sandstone', 'Marble', 'Limestone/Dolomite', 'Limestone/ Dolomite'],
    'Sedimentary Soft Rock': ['Limestone', 'Limestone with Shale', 'Miliolitic Limestone', 'Shale', 'Shale with Limestone', 'Shale with Sandstone', 'Shale, Limestone, and Sandstone', 'Laterite', 'Laterite/ Ferruginous Concretions'],
    'Volcanic Systems': ['Basalt', 'Granite'],
    'Himalayan Systems': ['Granite', 'Shale', 'Limestone', 'Schist', 'Gneiss', 'Quartzite'],
    'Alluvial': ['Alluvium'],
}

# Define the categories and the corresponding aquifers
crystalline_systems = ['Basalt', 'Granite', 'Schist', 'Quartzite', 'Charnockite', 'Basement Gneissic Complex', 'Gneiss', 'Intrusive', 'Khondalites']
sedimentary_hard_rock = ['Sandstone']
sedimentary_soft_rock = ['Limestone', 'Shale', 'Laterite']
volcanic_systems = ['Basalt', 'Granite']
himalayan_systems = ['Granite', 'Shale', 'Limestone', 'Schist', 'Gneiss', 'Quartzite']
alluvial = ['Alluvium']
unclassified=['Unclassified']
# State codes
volcanic_states = ['MS','MP','GJ','RJ','KA','JH','TG']
himalayan_states = ['AR', 'AS', 'MN', 'ML', 'MZ', 'NL', 'TR', 'SK','UK','LK','HP','JK']

# Create a function to classify aquifers
def classify_aquifer(row):
    aquifer = row['aquifer']
    state = row['state']

    if aquifer in volcanic_systems and state in volcanic_states:
        return 'Volcanic Systems'
    elif aquifer in himalayan_systems and state in himalayan_states:
        return 'Himalayan Systems'
    elif aquifer in crystalline_systems:
        return 'Crystalline Systems'
    elif aquifer in sedimentary_hard_rock:
        return 'Sedimentary Hard Rock'
    elif aquifer in sedimentary_soft_rock:
        return 'Sedimentary Soft Rock'
    elif aquifer in alluvial:
        return 'Alluvial'
    elif aquifer in unclassified:
        return 'Unclassified'
    else:
        return 'Others'

# Apply the classification
loaded_aquifer_gdf['aquifer_category'] = loaded_aquifer_gdf.apply(classify_aquifer, axis=1)

# Display the unique categories
print(loaded_aquifer_gdf['aquifer_category'].unique())

# Create a color map with unique colors for each aquifer category
unique_categories = loaded_aquifer_gdf['aquifer_category'].unique()
color_map = plt.cm.get_cmap('tab10', len(unique_categories))  # Using tab10 for vibrant colors

# Create a dictionary to map each category to a hex color
category_colors = {category: mcolors.to_hex(color_map(i)) for i, category in enumerate(unique_categories)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.2, label=category)  # Adjusted edge thickness

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Define the categories and the corresponding aquifers and their subtypes
aquifer_categories = {
    'Crystalline Systems': ['Basalt', 'Granite', 'Schist', 'Quartzite', 'Charnockite', 'Basement Gneissic Complex', 'Gneiss', 'Intrusive', 'Khondalites'],
    'Volcanic Systems': ['Basalt', 'Granite'],
    'Himalayan Systems': ['Granite','Shale','Limestone','Schist', 'Gneiss', 'Quartzite',],
    'Alluvial': ['Alluvium'],
    'Unclassified': ['Unclassified']
}

# Sedimentary rock subcategories
sedimentary_hard_rock_subcategories = ['Sandstone with Clay','Sandstone with Shale','Sandstone with shale/ coal beds','Sandstone/Conglomerate','Marble',
                                       'Shale/Shale with Sandstone', 'Shale, limestone and sandstone', 'Shale with Sandstone']
sedimentary_soft_rock_subcategories = ['Limestone/Dolomite', 'Limestone/ Dolomite', 'Limestone with Shale', 'Miliolitic Limestone','Shale',
                                        'Shale with limestone',
                                       'Shale with Limestone', 'Laterite/ Ferruginous concretions']

# State codes
volcanic_states = ['MS', 'MP', 'GJ', 'RJ', 'KA', 'JH', 'TG']
himalayan_states = ['AR', 'AS', 'MN', 'ML', 'MZ', 'NL', 'TR', 'SK', 'UK', 'LK', 'HP', 'JK']

# Create a function to classify aquifers
def classify_aquifer(row):
    aquifer = row['aquifer']
    aquifer0 = row['aquifer0']
    state = row['state']

    # Check for volcanic and Himalayan systems based on aquifer and state
    if aquifer in aquifer_categories['Volcanic Systems'] and state in volcanic_states:
        return 'Volcanic Systems'
    elif aquifer in aquifer_categories['Himalayan Systems'] and state in himalayan_states:
        return 'Himalayan Systems'

    # Check for sedimentary hard rock and soft rock based on aquifer and aquifer0
    elif aquifer in ['Sandstone','Limestone','Shale','Laterite']:
        if aquifer0 in sedimentary_hard_rock_subcategories:
            return 'Sedimentary Hard Rock'
        elif aquifer0 in sedimentary_soft_rock_subcategories:
            return 'Sedimentary Soft Rock'

    elif aquifer in aquifer_categories['Unclassified']:
        return 'Unclassified'
    # Check for other categories directly from aquifer
    for category, subtypes in aquifer_categories.items():
        if aquifer in subtypes:
            return category

    return 'Others'

# Load your GeoDataFrame
# loaded_aquifer_gdf = gpd.read_file('path_to_your_file.shp')

# Apply the classification
loaded_aquifer_gdf['aquifer_category'] = loaded_aquifer_gdf.apply(classify_aquifer, axis=1)

# Display the unique categories
print(loaded_aquifer_gdf['aquifer_category'].unique())

# Create a color map with unique colors for each aquifer category
unique_categories = loaded_aquifer_gdf['aquifer_category'].unique()
color_map = plt.cm.get_cmap('tab10', len(unique_categories))  # Using tab10 for vibrant colors

# Create a dictionary to map each category to a hex color
category_colors = {category: mcolors.to_hex(color_map(i)) for i, category in enumerate(unique_categories)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.2, label=category)  # Adjusted edge thickness

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
loaded_aquifer_gdf

In [ ]:
# Filter for the state Telangana
telangana_aquifers = loaded_aquifer_gdf[loaded_aquifer_gdf['state'] == 'TG']

# Further filter for sedimentary hard rocks and soft rocks
sedimentary_telangana_aquifers = telangana_aquifers[telangana_aquifers['aquifer_category'].isin(['Sedimentary Soft Rock', 'Sedimentary Hard Rock'])]


# Display the filtered DataFrame
sedimentary_telangana_aquifers[['aquifer', 'aquifer0','aquifers','aquifer_category','state']]

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Define the categories and the corresponding aquifers and their subtypes
aquifer_categories = {
    'Crystalline Systems': ['Basalt', 'Granite', 'Schist', 'Quartzite', 'Charnockite', 'Basement Gneissic Complex', 'Gneiss', 'Intrusive', 'Khondalites'],
    'Volcanic Systems': ['Basalt', 'Granite'],
    'Himalayan Systems': ['Granite','Shale','Limestone','Schist', 'Gneiss', 'Quartzite',],
    'Alluvial': ['Alluvium'],
    'Unclassified': ['Unclassified']
}

# Sedimentary rock subcategories
sedimentary_hard_rock_subcategories = ['Sandstone with Clay','Sandstone with Shale','Sandstone with shale/ coal beds','Sandstone/Conglomerate','Marble',
                                       'Shale/Shale with Sandstone', 'Shale, limestone and sandstone', 'Shale with Sandstone']
sedimentary_soft_rock_subcategories = ['Limestone/Dolomite', 'Limestone/ Dolomite', 'Limestone with Shale', 'Miliolitic Limestone','Shale',
                                        'Shale with limestone',
                                       'Shale with Limestone','Laterite/ Ferruginous concretions']

# State codes
volcanic_states = ['MS', 'MP', 'GJ', 'RJ', 'KA', 'JH', 'TG']
himalayan_states = ['AR', 'AS', 'MN', 'ML', 'MZ', 'NL', 'TR', 'SK', 'UK', 'LK', 'HP', 'JK']

# Create a function to classify aquifers
def classify_aquifer(row):
    aquifer = row['aquifer']
    aquifer0 = row['aquifer0']
    state = row['state']

    # Check for volcanic and Himalayan systems based on aquifer and state
    if aquifer in aquifer_categories['Volcanic Systems'] and state in volcanic_states:
        return 'Volcanic Systems'
    elif aquifer in aquifer_categories['Himalayan Systems'] and state in himalayan_states:
        return 'Himalayan Systems'

    # Check for sedimentary hard rock and soft rock based on aquifer and aquifer0
    elif aquifer in ['Sandstone','Limestone','Shale','Laterite']:
        if aquifer0 in sedimentary_hard_rock_subcategories:
            return 'Sedimentary Hard Rock'
        elif aquifer0 in sedimentary_soft_rock_subcategories:
            return 'Sedimentary Soft Rock'

    elif aquifer in aquifer_categories['Unclassified']:
        return 'Unclassified'
    # Check for other categories directly from aquifer
    for category, subtypes in aquifer_categories.items():
        if aquifer in subtypes:
            return category

    return 'Others'

# Load your GeoDataFrame
# loaded_aquifer_gdf = gpd.read_file('path_to_your_file.shp')

# Apply the classification
loaded_aquifer_gdf['aquifer_category'] = loaded_aquifer_gdf.apply(classify_aquifer, axis=1)

# Display the unique categories
print(loaded_aquifer_gdf['aquifer_category'].unique())

# Create a color map with unique colors for each aquifer category
unique_categories = loaded_aquifer_gdf['aquifer_category'].unique()
color_map = plt.cm.get_cmap('tab10', len(unique_categories))  # Using tab10 for vibrant colors

# Create a dictionary to map each category to a hex color
category_colors = {category: mcolors.to_hex(color_map(i)) for i, category in enumerate(unique_categories)}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.2, label=category)  # Adjusted edge thickness

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

# Color coding with the same colors from ACWADAM map

In [ ]:
# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    'Unclassified':'red',
    #'Others': 'gray'  # Adding a color for 'Other' category if it exists
}
# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category)  # Adjusted edge thickness

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Load your GeoDataFrame (assuming it's already loaded as 'loaded_aquifer_gdf')

# Ensure valid geometries and proper CRS
loaded_aquifer_gdf = loaded_aquifer_gdf[loaded_aquifer_gdf.is_valid]

# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    'Unclassified': 'red',
    #'Others': 'gray'  # Adding a color for 'Other' category if it exists
}

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))

for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category, aspect='equal')  # Adjusted edge thickness

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    #'Others': 'gray'  # Adding a color for 'Other' category if it exists
}
# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = loaded_aquifer_gdf[loaded_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category)

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

# Loading Microwatershed Data

In [ ]:
import pickle
import geopandas as gpd

# Path to the pickle file
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/micro_watershed_gdf.pickle'

# Load the GeoDataFrame from the pickle file
with open(pickle_path, 'rb') as f:
    loaded_micro_watershed_gdf = pickle.load(f)

# Verify the loaded GeoDataFrame
print(loaded_micro_watershed_gdf.head())

In [ ]:
loaded_micro_watershed_gdf

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Sample a smaller subset of the data
subset_gdf = loaded_micro_watershed_gdf.sample(n=30)  # Randomly select 30 microwatersheds

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black')

# Set the title and labels
ax.set_title('Sample of 30 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Display the plot
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Check the type of geometries
print(loaded_micro_watershed_gdf.geom_type.value_counts())

# Sample a smaller subset of the data
subset_gdf = loaded_micro_watershed_gdf.sample(n=30)  # Randomly select 30 microwatersheds

# Ensure the geometries are in the correct CRS for plotting (WGS84)
subset_gdf = subset_gdf.to_crs(epsg=4326)

# Check the type of geometries in the subset
print(subset_gdf.geom_type.value_counts())

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black')

# Set the title and labels
ax.set_title('Sample of 30 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Display the plot
plt.show()

In [ ]:
# Inspect the geometries of a few samples
sample_geometries = loaded_micro_watershed_gdf.sample(n=5)['geometry']
for i, geom in enumerate(sample_geometries):
    print(f"Geometry {i+1}: {geom}")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt


# Sample a single polygon
single_gdf = loaded_micro_watershed_gdf.sample(n=1)

# Ensure the geometry is in the correct CRS for plotting (WGS84)
single_gdf = single_gdf.to_crs(epsg=4326)

# Plot the single GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
single_gdf.boundary.plot(ax=ax, linewidth=1, color='black')  # Plot boundary of the polygon
single_gdf.plot(ax=ax, color='blue', alpha=0.5, edgecolor='black')  # Plot polygon with transparency

# Set the title and labels
ax.set_title('Single Microwatershed in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Display the plot
plt.show()

In [ ]:
# Check the current CRS
print(f"Current CRS: {loaded_micro_watershed_gdf.crs}")

In [ ]:
# Reproject to WGS84 if necessary
if loaded_micro_watershed_gdf.crs != 'EPSG:4326':
    reprojected_gdf = loaded_micro_watershed_gdf.to_crs(epsg=4326)
else:
    reprojected_gdf = loaded_micro_watershed_gdf

In [ ]:
# Sample a smaller subset of the data
subset_gdf = reprojected_gdf.sample(n=30)

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(12, 12))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)

# Set the title and labels
ax.set_title('Sample of 30 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Display the plot
plt.show()

In [ ]:
# Check the current CRS
print(f"Current CRS: {loaded_micro_watershed_gdf.crs}")

# Reproject to WGS84 if necessary
if loaded_micro_watershed_gdf.crs != 'EPSG:4326':
    reprojected_gdf = loaded_micro_watershed_gdf.to_crs(epsg=4326)
else:
    reprojected_gdf = loaded_micro_watershed_gdf

# Sample a smaller subset of the data
subset_gdf = reprojected_gdf.sample(n=5)

# Print the WKT of the geometries and their bounding boxes to inspect them
for i, geom in enumerate(subset_gdf['geometry']):
    print(f"Geometry {i+1} WKT: {geom.wkt}")
    print(f"Geometry {i+1} Bounds: {geom.bounds}")

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(12, 12))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)

# Set the title and labels
ax.set_title('Sample of 5 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Show coordinate extents
xlim = [subset_gdf.total_bounds[0], subset_gdf.total_bounds[2]]
ylim = [subset_gdf.total_bounds[1], subset_gdf.total_bounds[3]]
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# Display the plot
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Check the current CRS
print(f"Current CRS: {loaded_micro_watershed_gdf.crs}")

# Reproject to WGS84 if necessary
if loaded_micro_watershed_gdf.crs != 'EPSG:4326':
    reprojected_gdf = loaded_micro_watershed_gdf.to_crs(epsg=4326)
else:
    reprojected_gdf = loaded_micro_watershed_gdf

# Sample a smaller subset of the data
subset_gdf = reprojected_gdf.sample(n=30)

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(12, 12))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)

# Set the title and labels
ax.set_title('Sample of 30 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Adjust plot extents based on data bounds
xlim = [subset_gdf.total_bounds[0], subset_gdf.total_bounds[2]]
ylim = [subset_gdf.total_bounds[1], subset_gdf.total_bounds[3]]
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# Display the plot
plt.show()

In [ ]:
# Inspect the first few rows and geometry types
print(loaded_micro_watershed_gdf.head())
print(loaded_micro_watershed_gdf.geom_type)
print(loaded_micro_watershed_gdf.crs)

In [ ]:
# Sample a smaller subset of the data
subset_gdf = loaded_micro_watershed_gdf.sample(n=30)

# Plot the subset GeoDataFrame
fig, ax = plt.subplots(figsize=(12, 12))
subset_gdf.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)

# Set the title and labels
ax.set_title('Sample of 30 Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Adjust plot extents based on data bounds
xlim = [subset_gdf.total_bounds[0], subset_gdf.total_bounds[2]]
ylim = [subset_gdf.total_bounds[1], subset_gdf.total_bounds[3]]
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# Display the plot
plt.show()

In [ ]:
# Check if any geometries are invalid
invalid_geometries = loaded_micro_watershed_gdf[~loaded_micro_watershed_gdf.is_valid]

# Print the number of invalid geometries
print(f"Number of invalid geometries: {len(invalid_geometries)}")

# Print invalid geometries if any
if len(invalid_geometries) > 0:
    print(invalid_geometries)

In [ ]:
# Sample a smaller subset of the data
subset_gdf = loaded_micro_watershed_gdf.sample(n=30)

# Ensure the geometries are polygons
subset_gdf = subset_gdf[subset_gdf.geometry.type == 'Polygon']

# Create a plot with a base map
fig, ax = plt.subplots(figsize=(12, 12))

# Plot the microwatershed polygons
subset_gdf.plot(ax=ax, edgecolor='red', linewidth=1.5, facecolor='none')

# Set the extent to focus on India
ax.set_xlim([70, 90])
ax.set_ylim([5, 35])

# Add background features for better context
ax.set_title('Microwatersheds in India')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Add a grid for better reference
ax.grid(True)

# Display the plot
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import json
import pickle

In [ ]:
# Define the bounding box for the specified coordinates
bounding_box = box(30, 21, 100, 56)

# Filter the micro_watershed_gdf to include only geometries within the bounding box
filtered_micro_watershed_gdf = loaded_micro_watershed_gdf[loaded_micro_watershed_gdf.intersects(bounding_box)]

In [ ]:
filtered_micro_watershed_gdf.rename(columns={'id':'mws_id','geometry': 'mws_geometry'})

In [ ]:
# Select 30 random geometries from the filtered micro-watershed data
random_sample = loaded_micro_watershed_gdf.sample(n=3000, random_state=1)

# Plot the selected random points with transparency and reduced edge thickness
fig, ax = plt.subplots(figsize=(10, 10))
random_sample.plot(ax=ax, color='red', edgecolor='black', linewidth=0.2)

# Add title and axes labels
plt.title('Random Sample of 30 Micro-Watershed Geometries')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
filtered_micro_watershed_gdf.columns

In [ ]:
# Calculate the area of each geometry and sort by area in descending order
filtered_micro_watershed_gdf['area'] = filtered_micro_watershed_gdf.geometry.area
sorted_gdf = filtered_micro_watershed_gdf.sort_values(by='area', ascending=False)

# Select the top 30 geometries by area
top_30_gdf = sorted_gdf.head(30)

# Print the top 30 geometries
print(top_30_gdf[['id', 'area']])

# Plot the top 30 geometries with transparency and reduced edge thickness
fig, ax = plt.subplots(figsize=(10, 10))
top_30_gdf.plot(ax=ax, color='lightblue', edgecolor='black', linewidth=0.5)

# Add title and axes labels
plt.title('Top 30 Micro-Watershed Geometries by Area')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# Create a GeoDataFrame for wells
wells_gdf = gpd.GeoDataFrame(filtered_above_MP_gdf, geometry='geometry', crs="EPSG:4326")  # Use WGS 84

# Ensure both GeoDataFrames use the same CRS
wells_gdf = wells_gdf.to_crs(filtered_micro_watershed_gdf.crs)

# Perform a spatial join to associate each well with the corresponding microwatershed
joined_gdf = gpd.sjoin(wells_gdf, filtered_micro_watershed_gdf, how='left', op='within')

# Display the results
print(joined_gdf[['station_id', 'id']])  # 'id' is the identifier for the microwatershed

In [ ]:
joined_gdf

In [ ]:
# Merge to include the geometry of the microwatersheds
new_joined_gdf = joined_gdf.merge(filtered_micro_watershed_gdf[['id', 'geometry']], left_on='index_right', right_index=True, suffixes=('', '_micro_watershed'))

In [ ]:
new_joined_gdf

In [ ]:
# Perform a merge with indicator to identify rows only in joined_gdf
comparison_df = pd.merge(joined_gdf, new_joined_gdf, how='left', indicator=True)

# Filter the rows that are only present in joined_gdf
only_in_joined_gdf = comparison_df[comparison_df['_merge'] == 'left_only']

# Drop the indicator column
only_in_joined_gdf = only_in_joined_gdf.drop(columns=['_merge'])

only_in_joined_gdf

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
unique_msw=joined_gdf['id'].unique()

In [ ]:
print(len(unique_msw))

In [ ]:
joined_gdf.columns

In [ ]:
import pickle

In [ ]:
# # Save the joined_gdf as a pickle file
# pickle_path = '[private path removed]
# with open(pickle_path, 'wb') as f:
#     pickle.dump(joined_gdf, f)

# print(f"Joined GeoDataFrame saved as pickle at {pickle_path}")

In [ ]:
import pickle

In [ ]:
# Save the joined_gdf as a pickle file
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/well_with_containing_mws.pickle'
# Load the GeoDataFrame from the pickle file
with open(pickle_path, 'rb') as f:
    loaded_gdf = pickle.load(f)

# Verify the loaded GeoDataFrame
print(loaded_gdf.head())

In [ ]:
loaded_gdf

In [ ]:
loaded_gdf.columns

In [ ]:
for column in loaded_gdf.columns:
  print(column)

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import json

# Load the GeoDataFrame from the pickle file (assuming this step is already done)
# with open(pickle_path, 'rb') as f:
#     loaded_gdf = pickle.load(f)

# Assuming 'loaded_gdf' is your GeoDataFrame with well data and geometries

# Filter the GeoDataFrame to include only wells within the specified coordinates
filtered_gdf = loaded_gdf[(loaded_gdf.geometry.y >= 21) & (loaded_gdf.geometry.y <= 56) &
                          (loaded_gdf.geometry.x >= 30) & (loaded_gdf.geometry.x <= 100)]

# Verify the filtered DataFrame
print(filtered_gdf.head())

# Plot the filtered GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10))
filtered_gdf.plot(ax=ax, marker='o', color='blue', markersize=2, edgecolor='none', label='Well')

# Add title and axes labels
plt.title('Map of Well Locations within Specified Coordinates')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.legend()
plt.show()

In [ ]:
# Filter the GeoDataFrame to include only wells with NaN values in 'geometry_micro_watershed'
wells_with_nan_mws = loaded_gdf[loaded_gdf['index_right'].isna()]

wells_with_nan_mws = wells_with_nan_mws[(loaded_gdf.geometry.y >= 21) & (loaded_gdf.geometry.y <= 56) &
                          (loaded_gdf.geometry.x >= 30) & (loaded_gdf.geometry.x <= 100)]

# Print the wells with NaN values in 'geometry_micro_watershed'
print(wells_with_nan_mws[['objectid', 'station_id', 'geometry']])

In [ ]:
# Filter the GeoDataFrame to include only wells within the specified coordinates
latest_filtered_gdf = loaded_gdf[(loaded_gdf.geometry.y >= 21) & (loaded_gdf.geometry.y <= 56) &
                          (loaded_gdf.geometry.x >= 30) & (loaded_gdf.geometry.x <= 100)]


# Remove rows with NaN values in 'geometry_micro_watershed'
latest_filtered_gdf = latest_filtered_gdf[filtered_gdf['id'].notna()]

new_latest_filtered_gdf = latest_filtered_gdf.merge(filtered_micro_watershed_gdf[['id', 'geometry']], left_on='index_right', right_index=True, suffixes=('', '_micro_watershed'))

# Verify the filtered DataFrame
print(new_latest_filtered_gdf.head())

In [ ]:
# Create a GeoDataFrame for the microwatershed geometries
micro_watershed_gdf = gpd.GeoDataFrame(new_latest_filtered_gdf, geometry='geometry_micro_watershed')

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Plot wells
filtered_gdf.plot(ax=axes[0], marker='o', color='blue', markersize=2, edgecolor='black', label='Well')
axes[0].set_title('Map of Well Locations within Specified Coordinates')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Plot microwatershed polygons
micro_watershed_gdf.plot(ax=axes[1], facecolor='red', edgecolor='purple', linewidth=0.5, label='Microwatershed')
axes[1].set_title('Map of Microwatersheds within Specified Coordinates')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

# Add legends
axes[0].legend()
axes[1].legend()

# Show the plots
plt.tight_layout()
plt.show()

In [ ]:
new_latest_filtered_gdf

In [ ]:
unique_mws=new_latest_filtered_gdf['id'].unique()
print(len(unique_mws))

In [ ]:
# Group by 'id_micro_watershed' and retain the station with the highest consecutive_non_nan_pairs
new_latest_filtered_gdf_with_one_well_per_mws = new_latest_filtered_gdf.loc[new_latest_filtered_gdf.groupby('id_micro_watershed')['consecutive_non_nan_pairs'].idxmax()]

# Verify the filtered DataFrame
print(new_latest_filtered_gdf_with_one_well_per_mws[['objectid', 'station_id', 'id_micro_watershed', 'consecutive_non_nan_pairs', 'geometry_micro_watershed']])

In [ ]:
for column in new_latest_filtered_gdf_with_one_well_per_mws.columns:
  print(column)

In [ ]:
new_latest_filtered_gdf_with_one_well_per_mws

In [ ]:
# Rename conflicting columns in loaded_gdf if they exist
if 'index_left' in new_latest_filtered_gdf_with_one_well_per_mws.columns:
    new_latest_filtered_gdf_with_one_well_per_mws = new_latest_filtered_gdf_with_one_well_per_mws.rename(columns={'index_left': 'index_left_well'})
if 'index_right' in new_latest_filtered_gdf_with_one_well_per_mws:
    new_latest_filtered_gdf_with_one_well_per_mws = new_latest_filtered_gdf_with_one_well_per_mws.rename(columns={'index_right': 'index_right_mws'})

In [ ]:
# Ensure both GeoDataFrames use the same CRS
crs_new_latest_filtered_gdf_with_one_well_per_mws = new_latest_filtered_gdf_with_one_well_per_mws.to_crs(loaded_aquifer_gdf.crs)

# Verify that 'loaded_gdf' contains well points
print("Loaded GeoDataFrame (well points) head:")
print(crs_new_latest_filtered_gdf_with_one_well_per_mws.head())

In [ ]:
# Perform a spatial join to associate each well with the corresponding aquifer
final_gdf = gpd.sjoin(crs_new_latest_filtered_gdf_with_one_well_per_mws, loaded_aquifer_gdf, how='left', predicate='within')

# Display the results with relevant columns
print(final_gdf[['station_id', 'aquifer', 'aquifer_category']])

In [ ]:
for column in final_gdf.columns:
  print(column)

In [ ]:
# Merge to include the geometry of the microwatersheds
new_final_gdf = final_gdf.merge(loaded_aquifer_gdf[['id', 'geometry']], left_on='index_right', right_index=True, suffixes=('', '_aquifer'))

In [ ]:
new_final_gdf

In [ ]:
for column in new_final_gdf.columns:
  print(column)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Assuming 'cgwb_2001_onwards_renamed' is your dataframe
# The results dataframe already contains the necessary data
results = new_final_gdf[['station_id', 'consecutive_non_nan_pairs']]

# Calculate mean, median, and mode
mean_value = results['consecutive_non_nan_pairs'].mean()
median_value = results['consecutive_non_nan_pairs'].median()

print(f"Mean: {mean_value}")
print(f"Median: {median_value}")

# Calculate quartiles and percentiles
Q1 = results['consecutive_non_nan_pairs'].quantile(0.25)
Q3 = results['consecutive_non_nan_pairs'].quantile(0.75)
P10 = results['consecutive_non_nan_pairs'].quantile(0.10)
P20 = results['consecutive_non_nan_pairs'].quantile(0.20)

# Calculate the number of wells for each percentile and quartile
num_wells_P10 = results[results['consecutive_non_nan_pairs'] <= P10].shape[0]
num_wells_P20 = results[results['consecutive_non_nan_pairs'] <= P20].shape[0]
num_wells_Q1 = results[results['consecutive_non_nan_pairs'] <= Q1].shape[0]
num_wells_median = results[results['consecutive_non_nan_pairs'] <= median_value].shape[0]
num_wells_Q3 = results[results['consecutive_non_nan_pairs'] <= Q3].shape[0]

# Total number of wells
total_wells = results.shape[0]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(results['consecutive_non_nan_pairs'], bins=range(0, max(results['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
plt.title('Histogram of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.xlabel('Number of consecutive Non zero pair of values')
plt.ylabel('Number of Wells')
plt.xticks(range(0, max(results['consecutive_non_nan_pairs']) + 5, 5))
plt.grid(axis='y')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                         horizontalalignment='right', fontsize=10)
plt.show()

# Plot inverted box plot
plt.figure(figsize=(10, 6))
sns.boxplot(y=results['consecutive_non_nan_pairs'], color='lightyellow')
plt.title('Box Plot of number of consecutive Non zero pair of values present in the CGWB data (2001-2023)')
plt.ylabel('Number of consecutive Non zero pair of values')

# Annotate the quartiles and percentiles
plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

# Remove x-axis label as it's not used in the plot
plt.xlabel('')
plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                         f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                         horizontalalignment='right', fontsize=10)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'final_gdf' is your dataframe containing the necessary data
results = new_final_gdf[['station_id', 'consecutive_non_nan_pairs', 'aquifer_category']]

# Get the unique aquifer categories
unique_aquifer_categories = results['aquifer_category'].unique()

# Loop through each aquifer category and plot the histograms and box plots
for category in unique_aquifer_categories:
    category_data = results[results['aquifer_category'] == category]

    # Calculate mean, median, and quartiles
    mean_value = category_data['consecutive_non_nan_pairs'].mean()
    median_value = category_data['consecutive_non_nan_pairs'].median()
    Q1 = category_data['consecutive_non_nan_pairs'].quantile(0.25)
    Q3 = category_data['consecutive_non_nan_pairs'].quantile(0.75)
    P10 = category_data['consecutive_non_nan_pairs'].quantile(0.10)
    P20 = category_data['consecutive_non_nan_pairs'].quantile(0.20)

    # Calculate the number of wells for each percentile and quartile
    num_wells_P10 = category_data[category_data['consecutive_non_nan_pairs'] <= P10].shape[0]
    num_wells_P20 = category_data[category_data['consecutive_non_nan_pairs'] <= P20].shape[0]
    num_wells_Q1 = category_data[category_data['consecutive_non_nan_pairs'] <= Q1].shape[0]
    num_wells_median = category_data[category_data['consecutive_non_nan_pairs'] <= median_value].shape[0]
    num_wells_Q3 = category_data[category_data['consecutive_non_nan_pairs'] <= Q3].shape[0]

    # Total number of wells
    total_wells = category_data.shape[0]

    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(category_data['consecutive_non_nan_pairs'], bins=range(0, max(category_data['consecutive_non_nan_pairs']) + 5, 5), edgecolor='black')
    plt.title(f'Histogram of number of consecutive Non zero pair of values\nin the CGWB data (2001-2023) for {category}')
    plt.xlabel('Number of consecutive Non zero pair of values')
    plt.ylabel('Number of Wells')
    plt.xticks(range(0, max(category_data['consecutive_non_nan_pairs']) + 5, 5))
    plt.grid(axis='y')
    plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n',
                             horizontalalignment='right', fontsize=10)
    plt.show()

    # Plot box plot
    plt.figure(figsize=(10, 6))
    sns.boxplot(y=category_data['consecutive_non_nan_pairs'], color='lightyellow')
    plt.title(f'Box Plot of number of consecutive Non zero pair of values\nin the CGWB data (2001-2023) for {category}')
    plt.ylabel('Number of consecutive Non zero pair of values')

    # Annotate the quartiles and percentiles
    plt.text(0.8, P10, f'10th Percentile: {P10:.2f} ({num_wells_P10} wells)', horizontalalignment='center', verticalalignment='center', color='green')
    plt.text(0.8, P20, f'20th Percentile: {P20:.2f} ({num_wells_P20} wells)', horizontalalignment='center', verticalalignment='center', color='green')
    plt.text(0.8, Q1, f'Q1: {Q1:.2f} ({num_wells_Q1} wells)', horizontalalignment='center', verticalalignment='center', color='blue')
    plt.text(0.8, median_value, f'Median: {median_value:.2f} ({num_wells_median} wells)', horizontalalignment='center', verticalalignment='center', color='red')
    plt.text(0.8, Q3, f'Q3: {Q3:.2f} ({num_wells_Q3} wells)', horizontalalignment='center', verticalalignment='center', color='blue')

    # Remove x-axis label as it's not used in the plot
    plt.xlabel('')
    plt.figtext(0.99, 0.01, f'Total number of wells: {total_wells}\n'
                             f'The number of wells in brackets represents the number of wells whose data has the number of consecutive non-zero pairs of entries less than or equal to the value represented at the point.',
                             horizontalalignment='right', fontsize=10)
    plt.show()

In [ ]:
new_final_gdf

In [ ]:
for column in new_final_gdf.columns:
  print(column)

In [ ]:
# Filter out rows where aquifer is 'Unclassified'
new_final_filtered_gdf = new_final_gdf[new_final_gdf['aquifer_category'] != 'Unclassified']

# Compute the median value of consecutive_non_nan_pairs for 'Himalayan Systems'
himalayan_median = new_final_filtered_gdf[new_final_filtered_gdf['aquifer_category'] == 'Himalayan Systems']['consecutive_non_nan_pairs'].median()

# Apply the thresholds
new_final_filtered_gdf = new_final_filtered_gdf[
    ((new_final_filtered_gdf['aquifer_category'] != 'Himalayan Systems') & (new_final_filtered_gdf['consecutive_non_nan_pairs'] >= 70)) |
    ((new_final_filtered_gdf['aquifer_category'] == 'Himalayan Systems') & (new_final_filtered_gdf['consecutive_non_nan_pairs'] >= himalayan_median))
]

In [ ]:
print(himalayan_median)

In [ ]:
new_final_filtered_gdf

In [ ]:
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_after_threshold.pickle'
with open(pickle_path, 'wb') as f:
    pickle.dump(new_final_filtered_gdf, f)

print(f"Joined GeoDataFrame saved as pickle at {pickle_path}")

In [ ]:
import pickle

In [ ]:
# Save the joined_gdf as a pickle file
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_after_threshold.pickle'
# Load the GeoDataFrame from the pickle file
with open(pickle_path, 'rb') as f:
    ultimate_final_loaded_gdf = pickle.load(f)

# Verify the loaded GeoDataFrame
print(ultimate_final_loaded_gdf.head())

In [ ]:
ultimate_final_loaded_gdf

In [ ]:
for column in ultimate_final_loaded_gdf:
  print(column)

In [ ]:
shortented_final=ultimate_final_loaded_gdf[['id_micro_watershed','geometry_micro_watershed' ]]

In [ ]:
shortented_final

In [ ]:
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_mws_only_data_after_threshold.pickle'
with open(pickle_path, 'wb') as f:
    pickle.dump(shortented_final, f)

print(f"Joined GeoDataFrame saved as pickle at {pickle_path}")

In [ ]:
import geopandas as gpd

# Assuming ultimate_final_loaded_gdf is already loaded as a GeoDataFrame
# Select the desired columns and make a copy to avoid SettingWithCopyWarning
shortented_final = ultimate_final_loaded_gdf[['id_micro_watershed', 'geometry_micro_watershed']].copy()

# Convert the geometry column to WKT format if it's a geometry type using .loc
shortented_final.loc[:, 'geometry_micro_watershed'] = shortented_final['geometry_micro_watershed'].apply(lambda x: x.wkt if hasattr(x, 'wkt') else x)

# Define the path where you want to save the CSV file
csv_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_for_geometry_micro_watershed.csv'

# Save the DataFrame as a CSV file
shortented_final.to_csv(csv_path, index=False)

print(f"Joined GeoDataFrame saved as CSV at {csv_path}")

In [ ]:
shortented_final_stream_order=ultimate_final_loaded_gdf[['id_micro_watershed','station_id','station_name','.geo']]

In [ ]:
pickle_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_after_threshold_for_stream_order.pickle'
with open(pickle_path, 'wb') as f:
    pickle.dump(shortented_final_stream_order, f)

print(f"Joined GeoDataFrame saved as pickle at {pickle_path}")

In [ ]:
import geopandas as gpd

# Assuming ultimate_final_loaded_gdf is already loaded as a GeoDataFrame
# Select the desired columns
shortened_final_stream_order = ultimate_final_loaded_gdf[['id_micro_watershed', 'station_id', 'station_name', '.geo']].copy()

# Convert the geometry column (.geo) to WKT format if it's a geometry type
shortened_final_stream_order.loc[:, '.geo'] = shortened_final_stream_order['.geo'].apply(lambda x: x.wkt if hasattr(x, 'wkt') else x)

# Save the selected columns as a CSV file
shortened_final_stream_order.to_csv('shortened_final_stream_order.csv', index=False)

In [ ]:
# Define the path where you want to save the CSV file
csv_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_after_threshold_for_stream_order.csv'

# Save the DataFrame as a CSV file
shortened_final_stream_order.to_csv(csv_path, index=False)

print(f"Joined GeoDataFrame saved as CSV at {csv_path}")

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.colors as mcolors

# Assuming 'final_gdf' is your GeoDataFrame containing the aquifer data
# Define the color map for the aquifer categories
category_colors = {
    'Alluvial': 'sandybrown',
    'Himalayan Systems': 'peru',
    'Volcanic Systems': 'mediumseagreen',
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    #'Others': 'gray'  # Adding a color for 'Other' category if it exists
}

# Plot the aquifer categories
fig, ax = plt.subplots(figsize=(15, 10))
for category, color in category_colors.items():
    subset = ultimate_final_loaded_gdf[ultimate_final_loaded_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.2, label=category)

# Create custom legend
legend_handles = [plt.Line2D([0], [0], color=color, lw=4, label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    'Other': 'gray'  # Adding a color for 'Other' category if it exists
}
# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = ultimate_final_loaded_gdf[ultimate_final_loaded_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category)

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.patches import Patch

# Ensure 'geometry_aquifer' is set as the geometry column
# final_gdf = final_gdf.set_geometry('geometry_aquifer', drop=False)

# Define specific colors for each aquifer category
category_colors = {
    'Alluvial': '#FFDBAC',         # light sandlewood
    'Himalayan Systems': 'peru',       # mild yellow (skin color)
    'Volcanic Systems': 'mediumseagreen',  # medium green (like grass green)
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    'Other': 'gray'  # Adding a color for 'Other' category if it exists
}

# Create a GeoDataFrame for aquifer geometries
new_aquifer_gdf = ultimate_final_loaded_gdf

# Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
for category, color in category_colors.items():
    subset = new_aquifer_gdf[new_aquifer_gdf['aquifer_category'] == category]
    subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.1, label=category)

# Create custom legend
legend_handles = [Patch(facecolor=color, edgecolor='black', label=category) for category, color in category_colors.items()]

# Add the custom legend
ax.legend(handles=legend_handles, title='Aquifer Categories', loc='upper right', fontsize='small')

# Set title and axes labels
plt.title('Aquifer Map Based on Condensed Aquifer Types')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.patches as mpatches

# Count the occurrences of each aquifer category
category_counts = ultimate_final_loaded_gdf['aquifer_category'].value_counts()

# Define the color map for the aquifer categories
category_colors = {
    'Alluvial': 'sandybrown',
    'Himalayan Systems': 'peru',
    'Volcanic Systems': 'mediumseagreen',
    'Sedimentary Hard Rock': 'dodgerblue',
    'Sedimentary Soft Rock': 'skyblue',
    'Crystalline Systems': 'lightpink',
    #'Other': 'gray'  # Adding a color for 'Other' category if it exists
}

# Function to format the labels
def func(pct, allvals):
    absolute = int(pct/100.*sum(allvals))
    return "{:.1f}% ({:d})".format(pct, absolute)

# Plot the pie chart
fig, ax = plt.subplots(figsize=(10, 7))
wedges, texts, autotexts = ax.pie(category_counts, autopct=lambda pct: func(pct, category_counts), colors=[category_colors[cat] for cat in category_counts.index], startangle=140)

# Equal aspect ratio ensures that pie is drawn as a circle.
ax.axis('equal')

# Increase the size of the text
plt.setp(autotexts, size=10, weight="bold")
plt.setp(texts, size=10)

# Adjust the position of the labels to avoid overlap
for i, text in enumerate(texts):
    text.set_horizontalalignment('center')
    text.set_verticalalignment('center')
    angle = (wedges[i].theta2 - wedges[i].theta1)/2. + wedges[i].theta1
    x = wedges[i].r * 0.6 * (1 if angle < 180 else -1)
    y = wedges[i].r * 0.6 * (1 if angle < 90 or angle > 270 else -1)
    text.set_position((x, y))

adjust_text(autotexts, ax=ax)

# Add legend
handles = [mpatches.Patch(color=category_colors[cat], label=cat) for cat in category_counts.index]
ax.legend(handles=handles, title='Aquifer Categories', loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')

# Add title
plt.title('Distribution of Aquifer Categories after applying threshold')

# Show the plot
plt.show()

In [ ]:
# File Path (Adjust if needed)
file_path = f"{WORKSPACE_ROOT}/00. CGWB Ground Truth/input/combined_new_landforms_x_lulc.csv"

# Read CSV into a DataFrame
df = pd.read_csv(file_path)

In [ ]:
df

In [ ]:
for column in df.columns:
  print(column)

In [ ]:
# Inspect the .geo column
print(df['.geo'].head())

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

In [ ]:
# File Path (Adjust if needed)
file_path = f"{WORKSPACE_ROOT}/00. CGWB Ground Truth/input/final_filtered_data.csv"

# Read CSV into a DataFrame
final_filt_df = pd.read_csv(file_path)

In [ ]:
final_filt_df

In [ ]:
for column in final_filt_df.columns:
  print(column)

In [ ]:
# Assuming final_filt_df is your DataFrame
final_filt_df = final_filt_df.rename(columns={'pixel_valu': 'stream_order'})

# Verify the column name change
print(final_filt_df.columns)

In [ ]:
final_filt_df

In [ ]:
file_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/final_filtered_data_for_geometry_micro_watershed.csv'
# Read CSV into a DataFrame
terrain_filt_df = pd.read_csv(file_path)

In [ ]:
terrain_filt_df

In [ ]:
# Rename the column in terrain_filt_df to match final_filt_df
terrain_filt_df = terrain_filt_df.rename(columns={'id_micro_watershed': 'id_micro_w'})

# Perform the merge on 'id_micro_w'
terrain_stream_merged_df = pd.merge(final_filt_df, terrain_filt_df, on='id_micro_w')

# Display the first few rows of the merged DataFrame
print(terrain_stream_merged_df.head())

In [ ]:
for column in terrain_stream_merged_df.columns:
  print(column)

In [ ]:
terrain_stream_merged_df.head()

In [ ]:
import pandas as pd

# Assuming terrain_stream_merged_df is already created

# Drop the .geo_y column
terrain_stream_merged_df = terrain_stream_merged_df.drop(columns=['.geo_y'])

# Rename the .geo_x column to geo
terrain_stream_merged_df = terrain_stream_merged_df.rename(columns={'.geo_x': 'geo'})

# Display the updated columns
print("Updated columns:")
print(terrain_stream_merged_df.columns)

# Display the first few rows to verify the changes
print(terrain_stream_merged_df.head())

In [ ]:
# Perform the merge on 'id_micro_w' from terrain_stream_merged_df and 'id_micro_watershed' from ultimate_final_loaded_gdf
final_merged_df = pd.merge(terrain_stream_merged_df, ultimate_final_loaded_gdf, left_on='id_micro_w', right_on='id_micro_watershed')

# Display the first few rows of the final merged DataFrame
print(final_merged_df.head())

In [ ]:
final_merged_df

In [ ]:
for columns in final_merged_df.columns:
  print(columns)

In [ ]:
# Assuming final_merged_df is already created

# Count the number of rows for each stream_order type
stream_order_counts = final_merged_df.groupby('stream_order').size().reset_index(name='count')

# Display the counts for each stream_order type
print("Number of rows in each stream_order type:")
print(stream_order_counts)

In [ ]:
# Count the number of rows for each terrain_cluster type
terrain_cluster_counts = final_merged_df.groupby('terrain_cluster').size().reset_index(name='count')

# Display the counts for each terrain_cluster type
print("Number of rows in each terrain_cluster type:")
print(terrain_cluster_counts)

In [ ]:
# Check if the 'aquifer_category' column exists in the DataFrame
if 'aquifer_category' in final_merged_df.columns:
    # Get the unique aquifer categories
    unique_aquifer_categories = final_merged_df['aquifer_category'].unique()
    print("Unique aquifer categories in the final_merged_df:")
    for category in unique_aquifer_categories:
        print(category)
else:
    print("The 'aquifer_category' column does not exist in the final_merged_df.")

In [ ]:
import pandas as pd

# Get the unique aquifer categories
unique_aquifer_categories = final_merged_df['aquifer_category'].unique()

# Create a dictionary to store DataFrames for each aquifer category
aquifer_dfs = {}

# Iterate through each unique aquifer category and create a new DataFrame
for category in unique_aquifer_categories:
    # Filter the DataFrame based on the aquifer category
    aquifer_dfs[category] = final_merged_df[final_merged_df['aquifer_category'] == category]

    # Optionally, print the first few rows of each DataFrame to verify
    print(f"DataFrame for aquifer category: {category}")
    print(aquifer_dfs[category].head())


In [ ]:
aquifer_dfs

In [ ]:
output_dir = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/'
# Iterate through each unique aquifer category and save the relevant columns to CSV
for category in unique_aquifer_categories:
    # Filter the DataFrame based on the aquifer category
    category_df = final_merged_df[final_merged_df['aquifer_category'] == category]

    # Select the relevant columns
    category_df = category_df[['id_micro_watershed', 'geometry_micro_watershed']]

    # Define the output file path
    # Replace spaces with underscores for file names
    category_safe = category.replace(" ", "_")
    output_file = f"{output_dir}{category_safe}_aquifer.csv"

    # Save the DataFrame to a CSV file
    category_df.to_csv(output_file, index=False)

    print(f"Saved {category} aquifer data to {output_file}")

In [ ]:
# File Path (Adjust if needed)
file_path = f"{WORKSPACE_ROOT}/00. CGWB Ground Truth/input/WellDepths_combined_jun18_200.csv"

# Read CSV into a DataFrame
calculated_wd_df = pd.read_csv(file_path)

In [ ]:
calculated_wd_df

In [ ]:
for column in calculated_wd_df:
  print(column)

In [ ]:
# Extract the column '2001-01-01'
column_data = calculated_wd_df['2001-01-01']

# Print the first few values in the column
print("Values in the column '2001-01-01':")
for index, value in column_data.head().items():
    # Print the index and value
    print(f"Index: {index}, Value: {value}")

    # If the value is a string representing a dictionary, parse it
    try:
        parsed_value = json.loads(value)
        print(f"Parsed Value: {parsed_value}")
    except json.JSONDecodeError:
        print("Value is not a JSON string.")

In [ ]:
# List of columns to be updated
columns_to_update = [
    "2001-01-01", "2001-04-01", "2001-07-01", "2001-10-01",
    "2002-01-01", "2002-04-01", "2002-07-01", "2002-10-01",
    "2003-01-01", "2003-04-01", "2003-07-01", "2003-10-01",
    "2004-01-01", "2004-04-01", "2004-07-01", "2004-10-01",
    "2005-01-01", "2005-04-01", "2005-07-01", "2005-10-01",
    "2006-01-01", "2006-04-01", "2006-07-01", "2006-10-01",
    "2007-01-01", "2007-04-01", "2007-07-01", "2007-10-01",
    "2008-01-01", "2008-04-01", "2008-07-01", "2008-10-01",
    "2009-01-01", "2009-04-01", "2009-07-01", "2009-10-01",
    "2010-01-01", "2010-04-01", "2010-07-01", "2010-10-01",
    "2011-01-01", "2011-04-01", "2011-07-01", "2011-10-01",
    "2012-01-01", "2012-04-01", "2012-07-01", "2012-10-01",
    "2013-01-01", "2013-04-01", "2013-07-01", "2013-10-01",
    "2014-01-01", "2014-04-01", "2014-07-01", "2014-10-01",
    "2015-01-01", "2015-04-01", "2015-07-01", "2015-10-01",
    "2016-01-01", "2016-04-01", "2016-07-01", "2016-10-01",
    "2017-01-01", "2017-04-01", "2017-07-01", "2017-10-01",
    "2018-01-01", "2018-04-01", "2018-07-01", "2018-10-01",
    "2019-01-01", "2019-04-01", "2019-07-01", "2019-10-01",
    "2020-01-01", "2020-04-01", "2020-07-01", "2020-10-01",
    "2021-01-01", "2021-04-01", "2021-07-01", "2021-10-01",
    "2022-01-01", "2022-04-01", "2022-07-01", "2022-10-01",
    "2023-01-01", "2023-04-01", "2023-07-01"
]

# Function to extract "WellDepth" from dictionary
def extract_well_depth(value):
    try:
        parsed_value = json.loads(value)
        return parsed_value.get("WellDepth", None)
    except (json.JSONDecodeError, TypeError):
        return None

# Iterate through the columns and replace values with "WellDepth"
for column in columns_to_update:
    calculated_wd_df[column] = calculated_wd_df[column].apply(extract_well_depth)

# Display the first few rows to verify the changes
print(calculated_wd_df.head())

In [ ]:
calculated_wd_df

In [ ]:
import pandas as pd



# Create a dictionary mapping old column names to new column names
column_rename_map = {
    "2001-01-01": "2001_Jan_March_calculated",
    "2001-04-01": "2001_April_June_calculated",
    "2001-07-01": "2001_Jul_Sept_calculated",
    "2001-10-01": "2001_Oct_Dec_calculated",
    "2002-01-01": "2002_Jan_March_calculated",
    "2002-04-01": "2002_April_June_calculated",
    "2002-07-01": "2002_Jul_Sept_calculated",
    "2002-10-01": "2002_Oct_Dec_calculated",
    "2003-01-01": "2003_Jan_March_calculated",
    "2003-04-01": "2003_April_June_calculated",
    "2003-07-01": "2003_Jul_Sept_calculated",
    "2003-10-01": "2003_Oct_Dec_calculated",
    "2004-01-01": "2004_Jan_March_calculated",
    "2004-04-01": "2004_April_June_calculated",
    "2004-07-01": "2004_Jul_Sept_calculated",
    "2004-10-01": "2004_Oct_Dec_calculated",
    "2005-01-01": "2005_Jan_March_calculated",
    "2005-04-01": "2005_April_June_calculated",
    "2005-07-01": "2005_Jul_Sept_calculated",
    "2005-10-01": "2005_Oct_Dec_calculated",
    "2006-01-01": "2006_Jan_March_calculated",
    "2006-04-01": "2006_April_June_calculated",
    "2006-07-01": "2006_Jul_Sept_calculated",
    "2006-10-01": "2006_Oct_Dec_calculated",
    "2007-01-01": "2007_Jan_March_calculated",
    "2007-04-01": "2007_April_June_calculated",
    "2007-07-01": "2007_Jul_Sept_calculated",
    "2007-10-01": "2007_Oct_Dec_calculated",
    "2008-01-01": "2008_Jan_March_calculated",
    "2008-04-01": "2008_April_June_calculated",
    "2008-07-01": "2008_Jul_Sept_calculated",
    "2008-10-01": "2008_Oct_Dec_calculated",
    "2009-01-01": "2009_Jan_March_calculated",
    "2009-04-01": "2009_April_June_calculated",
    "2009-07-01": "2009_Jul_Sept_calculated",
    "2009-10-01": "2009_Oct_Dec_calculated",
    "2010-01-01": "2010_Jan_March_calculated",
    "2010-04-01": "2010_April_June_calculated",
    "2010-07-01": "2010_Jul_Sept_calculated",
    "2010-10-01": "2010_Oct_Dec_calculated",
    "2011-01-01": "2011_Jan_March_calculated",
    "2011-04-01": "2011_April_June_calculated",
    "2011-07-01": "2011_Jul_Sept_calculated",
    "2011-10-01": "2011_Oct_Dec_calculated",
    "2012-01-01": "2012_Jan_March_calculated",
    "2012-04-01": "2012_April_June_calculated",
    "2012-07-01": "2012_Jul_Sept_calculated",
    "2012-10-01": "2012_Oct_Dec_calculated",
    "2013-01-01": "2013_Jan_March_calculated",
    "2013-04-01": "2013_April_June_calculated",
    "2013-07-01": "2013_Jul_Sept_calculated",
    "2013-10-01": "2013_Oct_Dec_calculated",
    "2014-01-01": "2014_Jan_March_calculated",
    "2014-04-01": "2014_April_June_calculated",
    "2014-07-01": "2014_Jul_Sept_calculated",
    "2014-10-01": "2014_Oct_Dec_calculated",
    "2015-01-01": "2015_Jan_March_calculated",
    "2015-04-01": "2015_April_June_calculated",
    "2015-07-01": "2015_Jul_Sept_calculated",
    "2015-10-01": "2015_Oct_Dec_calculated",
    "2016-01-01": "2016_Jan_March_calculated",
    "2016-04-01": "2016_April_June_calculated",
    "2016-07-01": "2016_Jul_Sept_calculated",
    "2016-10-01": "2016_Oct_Dec_calculated",
    "2017-01-01": "2017_Jan_March_calculated",
    "2017-04-01": "2017_April_June_calculated",
    "2017-07-01": "2017_Jul_Sept_calculated",
    "2017-10-01": "2017_Oct_Dec_calculated",
    "2018-01-01": "2018_Jan_March_calculated",
    "2018-04-01": "2018_April_June_calculated",
    "2018-07-01": "2018_Jul_Sept_calculated",
    "2018-10-01": "2018_Oct_Dec_calculated",
    "2019-01-01": "2019_Jan_March_calculated",
    "2019-04-01": "2019_April_June_calculated",
    "2019-07-01": "2019_Jul_Sept_calculated",
    "2019-10-01": "2019_Oct_Dec_calculated",
    "2020-01-01": "2020_Jan_March_calculated",
    "2020-04-01": "2020_April_June_calculated",
    "2020-07-01": "2020_Jul_Sept_calculated",
    "2020-10-01": "2020_Oct_Dec_calculated",
    "2021-01-01": "2021_Jan_March_calculated",
    "2021-04-01": "2021_April_June_calculated",
    "2021-07-01": "2021_Jul_Sept_calculated",
    "2021-10-01": "2021_Oct_Dec_calculated",
    "2022-01-01": "2022_Jan_March_calculated",
    "2022-04-01": "2022_April_June_calculated",
    "2022-07-01": "2022_Jul_Sept_calculated",
    "2022-10-01": "2022_Oct_Dec_calculated",
    "2023-01-01": "2023_Jan_March_calculated",
    "2023-04-01": "2023_April_June_calculated",
    "2023-07-01": "2023_Jul_Sept_calculated",
    "2023-10-01": "2023_Oct_Dec_calculated"
}

# Rename the columns using the rename method
calculated_wd_df = calculated_wd_df.rename(columns=column_rename_map)

# Display the first few rows to verify the changes
print(calculated_wd_df.head())

In [ ]:
import pandas as pd

# Assuming calculated_wd_df and final_merged_df are already loaded

# Perform the merge on 'id_micro_watershed'
merged_microwatersheds_df = pd.merge(calculated_wd_df, final_merged_df, on='id_micro_watershed', how='inner')

# Display the first few rows of the merged DataFrame to verify the merge
print(merged_microwatersheds_df.head())

In [ ]:
# Perform the merge on 'id_micro_watershed', adding suffixes to overlapping column names
merged_microwatersheds_df = pd.merge(calculated_wd_df, final_merged_df, on='id_micro_watershed', how='inner', suffixes=('_calculated', '_final'))

# Display the first few rows of the merged DataFrame to verify the merge
print(merged_microwatersheds_df.head())

In [ ]:
merged_microwatersheds_df

In [ ]:
# Group by the 'aquifer_category' column and count the number of elements in each group
aquifer_counts = merged_microwatersheds_df.groupby('aquifer_category').size().reset_index(name='count')

# Display the counts
print(aquifer_counts)

In [ ]:
for columns in merged_microwatersheds_df.columns:
  print(columns)

In [ ]:
import pandas as pd

# Assuming merged_microwatersheds_df is already created

# Define the output file path
output_file_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/mainly_alluvial.pickle'

# Save the DataFrame as a pickle file
merged_microwatersheds_df.to_pickle(output_file_path)

print(f"Merged microwatersheds DataFrame saved to {output_file_path}")

In [ ]:
import pandas as pd

# Define the path to the pickle file
pickle_file_path = f'{WORKSPACE_ROOT}/output/mainly_alluvial.pickle'

# Read the pickle file into a DataFrame
first_df = pd.read_pickle(pickle_file_path)

# Display the first few rows of the loaded DataFrame to verify
print(first_df.head())

In [ ]:
for column in first_df.columns:
  print(column)

In [ ]:
import pandas as pd

# Assuming first_df is already loaded as a DataFrame

# List of seasons
seasons = [
    "2001_Jan_March", "2001_April_June", "2001_Jul_Sept", "2001_Oct_Dec",
    "2002_Jan_March", "2002_April_June", "2002_Jul_Sept", "2002_Oct_Dec",
    "2003_Jan_March", "2003_April_June", "2003_Jul_Sept", "2003_Oct_Dec",
    "2004_Jan_March", "2004_April_June", "2004_Jul_Sept", "2004_Oct_Dec",
    "2005_Jan_March", "2005_April_June", "2005_Jul_Sept", "2005_Oct_Dec",
    "2006_Jan_March", "2006_April_June", "2006_Jul_Sept", "2006_Oct_Dec",
    "2007_Jan_March", "2007_April_June", "2007_Jul_Sept", "2007_Oct_Dec",
    "2008_Jan_March", "2008_April_June", "2008_Jul_Sept", "2008_Oct_Dec",
    "2009_Jan_March", "2009_April_June", "2009_Jul_Sept", "2009_Oct_Dec",
    "2010_Jan_March", "2010_April_June", "2010_Jul_Sept", "2010_Oct_Dec",
    "2011_Jan_March", "2011_April_June", "2011_Jul_Sept", "2011_Oct_Dec",
    "2012_Jan_March", "2012_April_June", "2012_Jul_Sept", "2012_Oct_Dec",
    "2013_Jan_March", "2013_April_June", "2013_Jul_Sept", "2013_Oct_Dec",
    "2014_Jan_March", "2014_April_June", "2014_Jul_Sept", "2014_Oct_Dec",
    "2015_Jan_March", "2015_April_June", "2015_Jul_Sept", "2015_Oct_Dec",
    "2016_Jan_March", "2016_April_June", "2016_Jul_Sept", "2016_Oct_Dec",
    "2017_Jan_March", "2017_April_June", "2017_Jul_Sept", "2017_Oct_Dec",
    "2018_Jan_March", "2018_April_June", "2018_Jul_Sept", "2018_Oct_Dec",
    "2019_Jan_March", "2019_April_June", "2019_Jul_Sept", "2019_Oct_Dec",
    "2020_Jan_March", "2020_April_June", "2020_Jul_Sept", "2020_Oct_Dec",
    "2021_Jan_March", "2021_April_June", "2021_Jul_Sept", "2021_Oct_Dec",
    "2022_Jan_March", "2022_April_June", "2022_Jul_Sept", "2022_Oct_Dec",
    "2023_Jan_March", "2023_April_June", "2023_Jul_Sept", "2023_Oct_Dec"
]

# Dictionary to hold the new columns
new_columns = {}

# Iterate over the seasons to calculate the change in well depth
for i in range(len(seasons) - 1):
    current_season = seasons[i]
    next_season = seasons[i + 1]
    new_column_name = f"{current_season}_GT_CWD"

    # Calculate the change and handle NaN values
    new_columns[new_column_name] = first_df.apply(
        lambda row: row[next_season] - row[current_season] if pd.notna(row[next_season]) and pd.notna(row[current_season]) else float('nan'), axis=1
    )

# Combine the new columns with the original DataFrame
new_columns_df = pd.DataFrame(new_columns)
first_df = pd.concat([first_df, new_columns_df], axis=1)

# Display the first few rows to verify the changes
print(first_df.head())

In [ ]:
first_df

In [ ]:
for column in first_df.columns:
  print(column)

In [ ]:
import pandas as pd

# Assuming first_df is already loaded as a DataFrame

# List of columns to retain
columns_to_retain = [
    "2001_Jan_March_calculated", "2001_April_June_calculated", "2001_Jul_Sept_calculated", "2001_Oct_Dec_calculated",
    "2002_Jan_March_calculated", "2002_April_June_calculated", "2002_Jul_Sept_calculated", "2002_Oct_Dec_calculated",
    "2003_Jan_March_calculated", "2003_April_June_calculated", "2003_Jul_Sept_calculated", "2003_Oct_Dec_calculated",
    "2004_Jan_March_calculated", "2004_April_June_calculated", "2004_Jul_Sept_calculated", "2004_Oct_Dec_calculated",
    "2005_Jan_March_calculated", "2005_April_June_calculated", "2005_Jul_Sept_calculated", "2005_Oct_Dec_calculated",
    "2006_Jan_March_calculated", "2006_April_June_calculated", "2006_Jul_Sept_calculated", "2006_Oct_Dec_calculated",
    "2007_Jan_March_calculated", "2007_April_June_calculated", "2007_Jul_Sept_calculated", "2007_Oct_Dec_calculated",
    "2008_Jan_March_calculated", "2008_April_June_calculated", "2008_Jul_Sept_calculated", "2008_Oct_Dec_calculated",
    "2009_Jan_March_calculated", "2009_April_June_calculated", "2009_Jul_Sept_calculated", "2009_Oct_Dec_calculated",
    "2010_Jan_March_calculated", "2010_April_June_calculated", "2010_Jul_Sept_calculated", "2010_Oct_Dec_calculated",
    "2011_Jan_March_calculated", "2011_April_June_calculated", "2011_Jul_Sept_calculated", "2011_Oct_Dec_calculated",
    "2012_Jan_March_calculated", "2012_April_June_calculated", "2012_Jul_Sept_calculated", "2012_Oct_Dec_calculated",
    "2013_Jan_March_calculated", "2013_April_June_calculated", "2013_Jul_Sept_calculated", "2013_Oct_Dec_calculated",
    "2014_Jan_March_calculated", "2014_April_June_calculated", "2014_Jul_Sept_calculated", "2014_Oct_Dec_calculated",
    "2015_Jan_March_calculated", "2015_April_June_calculated", "2015_Jul_Sept_calculated", "2015_Oct_Dec_calculated",
    "2016_Jan_March_calculated", "2016_April_June_calculated", "2016_Jul_Sept_calculated", "2016_Oct_Dec_calculated",
    "2017_Jan_March_calculated", "2017_April_June_calculated", "2017_Jul_Sept_calculated", "2017_Oct_Dec_calculated",
    "2018_Jan_March_calculated", "2018_April_June_calculated", "2018_Jul_Sept_calculated", "2018_Oct_Dec_calculated",
    "2019_Jan_March_calculated", "2019_April_June_calculated", "2019_Jul_Sept_calculated", "2019_Oct_Dec_calculated",
    "2020_Jan_March_calculated", "2020_April_June_calculated", "2020_Jul_Sept_calculated", "2020_Oct_Dec_calculated",
    "2021_Jan_March_calculated", "2021_April_June_calculated", "2021_Jul_Sept_calculated", "2021_Oct_Dec_calculated",
    "2022_Jan_March_calculated", "2022_April_June_calculated", "2022_Jul_Sept_calculated", "2022_Oct_Dec_calculated",
    "2023_Jan_March_calculated", "2023_April_June_calculated", "2023_Jul_Sept_calculated",
    "id_micro_watershed", "geo", "stream_order", "terrain_cluster", "aquifer_category",
    "2001_Jan_March_GT_CWD", "2001_April_June_GT_CWD", "2001_Jul_Sept_GT_CWD", "2001_Oct_Dec_GT_CWD",
    "2002_Jan_March_GT_CWD", "2002_April_June_GT_CWD", "2002_Jul_Sept_GT_CWD", "2002_Oct_Dec_GT_CWD",
    "2003_Jan_March_GT_CWD", "2003_April_June_GT_CWD", "2003_Jul_Sept_GT_CWD", "2003_Oct_Dec_GT_CWD",
    "2004_Jan_March_GT_CWD", "2004_April_June_GT_CWD", "2004_Jul_Sept_GT_CWD", "2004_Oct_Dec_GT_CWD",
    "2005_Jan_March_GT_CWD", "2005_April_June_GT_CWD", "2005_Jul_Sept_GT_CWD", "2005_Oct_Dec_GT_CWD",
    "2006_Jan_March_GT_CWD", "2006_April_June_GT_CWD", "2006_Jul_Sept_GT_CWD", "2006_Oct_Dec_GT_CWD",
    "2007_Jan_March_GT_CWD", "2007_April_June_GT_CWD", "2007_Jul_Sept_GT_CWD", "2007_Oct_Dec_GT_CWD",
    "2008_Jan_March_GT_CWD", "2008_April_June_GT_CWD", "2008_Jul_Sept_GT_CWD", "2008_Oct_Dec_GT_CWD",
    "2009_Jan_March_GT_CWD", "2009_April_June_GT_CWD", "2009_Jul_Sept_GT_CWD", "2009_Oct_Dec_GT_CWD",
    "2010_Jan_March_GT_CWD", "2010_April_June_GT_CWD", "2010_Jul_Sept_GT_CWD", "2010_Oct_Dec_GT_CWD",
    "2011_Jan_March_GT_CWD", "2011_April_June_GT_CWD", "2011_Jul_Sept_GT_CWD", "2011_Oct_Dec_GT_CWD",
    "2012_Jan_March_GT_CWD", "2012_April_June_GT_CWD", "2012_Jul_Sept_GT_CWD", "2012_Oct_Dec_GT_CWD",
    "2013_Jan_March_GT_CWD", "2013_April_June_GT_CWD", "2013_Jul_Sept_GT_CWD", "2013_Oct_Dec_GT_CWD",
    "2014_Jan_March_GT_CWD", "2014_April_June_GT_CWD", "2014_Jul_Sept_GT_CWD", "2014_Oct_Dec_GT_CWD",
    "2015_Jan_March_GT_CWD", "2015_April_June_GT_CWD", "2015_Jul_Sept_GT_CWD", "2015_Oct_Dec_GT_CWD",
    "2016_Jan_March_GT_CWD", "2016_April_June_GT_CWD", "2016_Jul_Sept_GT_CWD", "2016_Oct_Dec_GT_CWD",
    "2017_Jan_March_GT_CWD", "2017_April_June_GT_CWD", "2017_Jul_Sept_GT_CWD", "2017_Oct_Dec_GT_CWD",
    "2018_Jan_March_GT_CWD", "2018_April_June_GT_CWD", "2018_Jul_Sept_GT_CWD", "2018_Oct_Dec_GT_CWD",
    "2019_Jan_March_GT_CWD", "2019_April_June_GT_CWD", "2019_Jul_Sept_GT_CWD", "2019_Oct_Dec_GT_CWD",
    "2020_Jan_March_GT_CWD", "2020_April_June_GT_CWD", "2020_Jul_Sept_GT_CWD", "2020_Oct_Dec_GT_CWD",
    "2021_Jan_March_GT_CWD", "2021_April_June_GT_CWD", "2021_Jul_Sept_GT_CWD", "2021_Oct_Dec_GT_CWD",
    "2022_Jan_March_GT_CWD", "2022_April_June_GT_CWD", "2022_Jul_Sept_GT_CWD", "2022_Oct_Dec_GT_CWD",
    "2023_Jan_March_GT_CWD", "2023_April_June_GT_CWD", "2023_Jul_Sept_GT_CWD"
]

# Retain only the specified columns
first_df = first_df[columns_to_retain]

# Display the first few rows to verify the changes
print(first_df.head())

In [ ]:
for column in first_df.columns:
  print(column)

In [ ]:
import pandas as pd

# Assuming first_df is already loaded as a DataFrame

# List of GT change in well depth columns
gt_change_columns = [
    "2001_Jan_March_GT_CWD", "2001_April_June_GT_CWD", "2001_Jul_Sept_GT_CWD", "2001_Oct_Dec_GT_CWD",
    "2002_Jan_March_GT_CWD", "2002_April_June_GT_CWD", "2002_Jul_Sept_GT_CWD", "2002_Oct_Dec_GT_CWD",
    "2003_Jan_March_GT_CWD", "2003_April_June_GT_CWD", "2003_Jul_Sept_GT_CWD", "2003_Oct_Dec_GT_CWD",
    "2004_Jan_March_GT_CWD", "2004_April_June_GT_CWD", "2004_Jul_Sept_GT_CWD", "2004_Oct_Dec_GT_CWD",
    "2005_Jan_March_GT_CWD", "2005_April_June_GT_CWD", "2005_Jul_Sept_GT_CWD", "2005_Oct_Dec_GT_CWD",
    "2006_Jan_March_GT_CWD", "2006_April_June_GT_CWD", "2006_Jul_Sept_GT_CWD", "2006_Oct_Dec_GT_CWD",
    "2007_Jan_March_GT_CWD", "2007_April_June_GT_CWD", "2007_Jul_Sept_GT_CWD", "2007_Oct_Dec_GT_CWD",
    "2008_Jan_March_GT_CWD", "2008_April_June_GT_CWD", "2008_Jul_Sept_GT_CWD", "2008_Oct_Dec_GT_CWD",
    "2009_Jan_March_GT_CWD", "2009_April_June_GT_CWD", "2009_Jul_Sept_GT_CWD", "2009_Oct_Dec_GT_CWD",
    "2010_Jan_March_GT_CWD", "2010_April_June_GT_CWD", "2010_Jul_Sept_GT_CWD", "2010_Oct_Dec_GT_CWD",
    "2011_Jan_March_GT_CWD", "2011_April_June_GT_CWD", "2011_Jul_Sept_GT_CWD", "2011_Oct_Dec_GT_CWD",
    "2012_Jan_March_GT_CWD", "2012_April_June_GT_CWD", "2012_Jul_Sept_GT_CWD", "2012_Oct_Dec_GT_CWD",
    "2013_Jan_March_GT_CWD", "2013_April_June_GT_CWD", "2013_Jul_Sept_GT_CWD", "2013_Oct_Dec_GT_CWD",
    "2014_Jan_March_GT_CWD", "2014_April_June_GT_CWD", "2014_Jul_Sept_GT_CWD", "2014_Oct_Dec_GT_CWD",
    "2015_Jan_March_GT_CWD", "2015_April_June_GT_CWD", "2015_Jul_Sept_GT_CWD", "2015_Oct_Dec_GT_CWD",
    "2016_Jan_March_GT_CWD", "2016_April_June_GT_CWD", "2016_Jul_Sept_GT_CWD", "2016_Oct_Dec_GT_CWD",
    "2017_Jan_March_GT_CWD", "2017_April_June_GT_CWD", "2017_Jul_Sept_GT_CWD", "2017_Oct_Dec_GT_CWD",
    "2018_Jan_March_GT_CWD", "2018_April_June_GT_CWD", "2018_Jul_Sept_GT_CWD", "2018_Oct_Dec_GT_CWD",
    "2019_Jan_March_GT_CWD", "2019_April_June_GT_CWD", "2019_Jul_Sept_GT_CWD", "2019_Oct_Dec_GT_CWD",
    "2020_Jan_March_GT_CWD", "2020_April_June_GT_CWD", "2020_Jul_Sept_GT_CWD", "2020_Oct_Dec_GT_CWD",
    "2021_Jan_March_GT_CWD", "2021_April_June_GT_CWD", "2021_Jul_Sept_GT_CWD", "2021_Oct_Dec_GT_CWD",
    "2022_Jan_March_GT_CWD", "2022_April_June_GT_CWD", "2022_Jul_Sept_GT_CWD", "2022_Oct_Dec_GT_CWD",
    "2023_Jan_March_GT_CWD", "2023_April_June_GT_CWD", "2023_Jul_Sept_GT_CWD"
]

# List of calculated well depth columns
calculated_columns = [
    "2001_Jan_March_calculated", "2001_April_June_calculated", "2001_Jul_Sept_calculated", "2001_Oct_Dec_calculated",
    "2002_Jan_March_calculated", "2002_April_June_calculated", "2002_Jul_Sept_calculated", "2002_Oct_Dec_calculated",
    "2003_Jan_March_calculated", "2003_April_June_calculated", "2003_Jul_Sept_calculated", "2003_Oct_Dec_calculated",
    "2004_Jan_March_calculated", "2004_April_June_calculated", "2004_Jul_Sept_calculated", "2004_Oct_Dec_calculated",
    "2005_Jan_March_calculated", "2005_April_June_calculated", "2005_Jul_Sept_calculated", "2005_Oct_Dec_calculated",
    "2006_Jan_March_calculated", "2006_April_June_calculated", "2006_Jul_Sept_calculated", "2006_Oct_Dec_calculated",
    "2007_Jan_March_calculated", "2007_April_June_calculated", "2007_Jul_Sept_calculated", "2007_Oct_Dec_calculated",
    "2008_Jan_March_calculated", "2008_April_June_calculated", "2008_Jul_Sept_calculated", "2008_Oct_Dec_calculated",
    "2009_Jan_March_calculated", "2009_April_June_calculated", "2009_Jul_Sept_calculated", "2009_Oct_Dec_calculated",
    "2010_Jan_March_calculated", "2010_April_June_calculated", "2010_Jul_Sept_calculated", "2010_Oct_Dec_calculated",
    "2011_Jan_March_calculated", "2011_April_June_calculated", "2011_Jul_Sept_calculated", "2011_Oct_Dec_calculated",
    "2012_Jan_March_calculated", "2012_April_June_calculated", "2012_Jul_Sept_calculated", "2012_Oct_Dec_calculated",
    "2013_Jan_March_calculated", "2013_April_June_calculated", "2013_Jul_Sept_calculated", "2013_Oct_Dec_calculated",
    "2014_Jan_March_calculated", "2014_April_June_calculated", "2014_Jul_Sept_calculated", "2014_Oct_Dec_calculated",
    "2015_Jan_March_calculated", "2015_April_June_calculated", "2015_Jul_Sept_calculated", "2015_Oct_Dec_calculated",
    "2016_Jan_March_calculated", "2016_April_June_calculated", "2016_Jul_Sept_calculated", "2016_Oct_Dec_calculated",
    "2017_Jan_March_calculated", "2017_April_June_calculated", "2017_Jul_Sept_calculated", "2017_Oct_Dec_calculated",
    "2018_Jan_March_calculated", "2018_April_June_calculated", "2018_Jul_Sept_calculated", "2018_Oct_Dec_calculated",
    "2019_Jan_March_calculated", "2019_April_June_calculated", "2019_Jul_Sept_calculated", "2019_Oct_Dec_calculated",
    "2020_Jan_March_calculated", "2020_April_June_calculated", "2020_Jul_Sept_calculated", "2020_Oct_Dec_calculated",
    "2021_Jan_March_calculated", "2021_April_June_calculated", "2021_Jul_Sept_calculated", "2021_Oct_Dec_calculated",
    "2022_Jan_March_calculated", "2022_April_June_calculated", "2022_Jul_Sept_calculated", "2022_Oct_Dec_calculated",
    "2023_Jan_March_calculated", "2023_April_June_calculated", "2023_Jul_Sept_calculated"
]

# Ensure that both lists have the same order and length
assert len(gt_change_columns) == len(calculated_columns)

# Create a list to hold the new rows
new_rows = []

# Iterate over each row in the original DataFrame
for idx, row in first_df.iterrows():
    # Iterate over each pair of GT change and calculated columns
    for gt_col, calc_col in zip(gt_change_columns, calculated_columns):
        new_row = {
            'id_micro_watershed': row['id_micro_watershed'],
            'geo': row['geo'],
            'stream_order': row['stream_order'],
            'terrain_cluster': row['terrain_cluster'],
            'aquifer_category': row['aquifer_category'],
            'season': gt_col.replace('_GT_CWD', ''),  # Extract season name
            'GT_change_in_well_depth': row[gt_col],
            'calculated_well_depth': row[calc_col]
        }
        new_rows.append(new_row)

# Create a new DataFrame from the new rows
new_df = pd.DataFrame(new_rows)

# Remove rows where either GT_change_in_well_depth or calculated_well_depth is NaN
cleaned_df = new_df.dropna(subset=['GT_change_in_well_depth', 'calculated_well_depth'])

# Display the first few rows of the cleaned DataFrame to verify the changes
print(cleaned_df.head())

In [ ]:
new_df

# Remove NaN values

In [ ]:
# Remove rows where either GT_change_in_well_depth or calculated_well_depth is NaN
cleaned_df = new_df.dropna(subset=['GT_change_in_well_depth', 'calculated_well_depth'])

# Display the first few rows of the cleaned DataFrame to verify the changes
print(cleaned_df)

# Select Alluvial

In [ ]:
# Filter the DataFrame to include only rows with aquifer_category as "Alluvial"
alluvial_df = cleaned_df.loc[cleaned_df['aquifer_category'] == 'Alluvial']

# Display the first few rows of the filtered DataFrame to verify the changes
print(alluvial_df.head())

In [ ]:
alluvial_df

In [ ]:
import pandas as pd

# Define the file path for reading the CSV
csv_file_path = f'{WORKSPACE_ROOT}/input/final_filtered_data_for_geo_mws_so_area.csv'

# Read the CSV file into a DataFrame
stream_order_continous = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame to ensure it was read correctly
print(stream_order_continous)


In [ ]:
# Merge the DataFrames based on the specified columns
alluvial_continous_stream_order = pd.merge(alluvial_df, stream_order_continous, left_on='id_micro_watershed', right_on='id_micro_w', how='inner')

# Display the first few rows of the merged DataFrame to ensure it was merged correctly
print(alluvial_continous_stream_order)

In [ ]:
for column in alluvial_continous_stream_order.columns:
  print(column)

In [ ]:
# Creating a copy of the original DataFrame
renamed_alluvial_continous_stream_order = alluvial_continous_stream_order.copy()

# Removing the columns 'id_micro_w' and 'geometry' from the new DataFrame
renamed_alluvial_continous_stream_order = renamed_alluvial_continous_stream_order.drop(columns=['id_micro_w', 'geometry'])

# Renaming 'stream_order' to 'stream_order_continous' in the new DataFrame
renamed_alluvial_continous_stream_order = renamed_alluvial_continous_stream_order.rename(columns={'stream_order': 'stream_order_categorical'})

# Renaming 'Area%_SO_n' to 'percentage_area_stream_order_n' in the new DataFrame
renamed_alluvial_continous_stream_order.columns = [
    col.replace('Area%_SO_', 'percentage_area_stream_order_') if col.startswith('Area%_SO_') else col
    for col in renamed_alluvial_continous_stream_order.columns
]

# Display the first few rows of the new DataFrame to ensure the changes were applied correctly
print(renamed_alluvial_continous_stream_order)

In [ ]:
import pandas as pd

# Assuming merged_microwatersheds_df is already created

# Define the output file path
output_file_path = f'{WORKSPACE_ROOT}/00. CGWB Ground Truth/output/only_alluvial.pickle'

# Save the DataFrame as a pickle file
alluvial_df.to_pickle(output_file_path)

print(f"Alluvial DataFrame saved to {output_file_path}")

In [ ]:
import pandas as pd

# Assuming merged_microwatersheds_df is already created

# Define the output file path
output_file_path = f'{WORKSPACE_ROOT}/output/only_alluvial_continous_stream_order.pickle'

# Save the DataFrame as a pickle file
renamed_alluvial_continous_stream_order.to_pickle(output_file_path)

print(f"Alluvial DataFrame saved to {output_file_path}")

In [ ]:
import pandas as pd

# Define the input file path
input_file_path = f'{WORKSPACE_ROOT}/output/only_alluvial_continous_stream_order.pickle'

# Load the DataFrame from the pickle file
loaded_alluvial_df = pd.read_pickle(input_file_path)

# Display the first few rows of the loaded DataFrame to verify
print(loaded_alluvial_df.head())

In [ ]:
# Print only the 'geo' column
print(loaded_alluvial_df['geo'])

In [ ]:
import json

# Function to extract coordinates from the geo JSON
def extract_coordinates(geo_json):
    geo_dict = json.loads(geo_json)
    return geo_dict['coordinates']

# Apply the function to the 'geo' column
loaded_alluvial_df['coordinates'] = loaded_alluvial_df['geo'].apply(extract_coordinates)

# Print the coordinates column
print(loaded_alluvial_df['coordinates'])

In [ ]:
loaded_alluvial_df

In [ ]:
import json

# Function to extract latitude and longitude from the geo JSON
def extract_lat_lon(geo_json):
    geo_dict = json.loads(geo_json)
    coordinates = geo_dict['coordinates']
    return coordinates[1], coordinates[0]  # Assuming the coordinates are in [longitude, latitude] order

# Apply the function to the 'geo' column and create new columns for latitude and longitude
loaded_alluvial_df[['latitude', 'longitude']] = loaded_alluvial_df['geo'].apply(lambda x: pd.Series(extract_lat_lon(x)))

# Print the DataFrame to verify the changes
print(loaded_alluvial_df[['latitude', 'longitude']])

In [ ]:
for column in loaded_alluvial_df.columns:
  print(column)

In [ ]:
# Define a mapping from season names to numeric values
season_mapping = {
    "Jan_March": 1,
    "April_June": 2,
    "Jul_Sept": 3,
    "Oct_Dec": 4
}

# Extract the season part from the 'season' column and map it to the numeric values
loaded_alluvial_df['season_numeric'] = loaded_alluvial_df['season'].str.extract(r'(\w+_\w+)', expand=False).replace(season_mapping)

# Verify the changes
print(loaded_alluvial_df[['season', 'season_numeric']].head())

In [ ]:
# Define a mapping from season names to numeric values
season_mapping = {
    "Jan_March": 1,
    "April_June": 2,
    "Jul_Sept": 3,
    "Oct_Dec": 4
}

# Extract the season part from the 'season' column and map it to the numeric values
loaded_alluvial_df['season_part'] = loaded_alluvial_df['season'].str.split('_').str[1] + '_' + loaded_alluvial_df['season'].str.split('_').str[2]
loaded_alluvial_df['season_numeric'] = loaded_alluvial_df['season_part'].replace(season_mapping)

# Verify the changes
print(loaded_alluvial_df[['season', 'season_part', 'season_numeric']])

In [ ]:
loaded_alluvial_df

In [ ]:
for column in loaded_alluvial_df.columns:
  print(column)

In [ ]:
# Count the number of rows where 'calculated_well_depth' is 0.0
zero_well_depth_count = (loaded_alluvial_df['calculated_well_depth'] == 0).sum()

# Print the result
print(f"Number of rows with calculated_well_depth as 0.0: {zero_well_depth_count}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features and target
features = [
    'stream_order_categorical', 'terrain_cluster', 'calculated_well_depth', 'latitude', 'longitude', 'season_numeric',
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11'
]
target = 'GT_change_in_well_depth'

without_lag_latest_loaded_alluvial_df = loaded_alluvial_df.copy()

# Drop NA values generated by lagging
without_lag_latest_loaded_alluvial_df = without_lag_latest_loaded_alluvial_df.dropna()

X = without_lag_latest_loaded_alluvial_df[features]
y = without_lag_latest_loaded_alluvial_df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
categorical_features_onehot = ['terrain_cluster']
categorical_features_ordinal = ['stream_order_categorical', 'season_numeric']
numerical_features = [
    'calculated_well_depth', 'latitude', 'longitude', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
    'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
    'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
    'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat_onehot', OneHotEncoder(), categorical_features_onehot),
        ('cat_ordinal', OrdinalEncoder(), categorical_features_ordinal),
        ('num', 'passthrough', numerical_features)  # Pass through numerical features without scaling
    ]
)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import statsmodels.api as sm
import numpy as np

# Assuming X is your dataset and preprocessor is your ColumnTransformer
X_transformed = preprocessor.fit_transform(X)

# Get feature names from the transformers in the preprocessor
onehot_columns = preprocessor.named_transformers_['cat_onehot'].get_feature_names_out()
ordinal_columns = preprocessor.named_transformers_['cat_ordinal'].get_feature_names_out()
# You may need to manually specify these if get_feature_names_out() does not apply

# Concatenate all feature names into a single array
feature_names = np.concatenate([onehot_columns, ordinal_columns, numerical_features])

# Create a DataFrame from the transformed data
X_transformed_df = pd.DataFrame(X_transformed, columns=feature_names)

# First split the transformed data into training and testing sets without a constant
X_train, X_test, y_train, y_test = train_test_split(X_transformed_df, y, test_size=0.2, random_state=42)

# Create copies for use with statsmodels by adding a constant to only the training and testing set
X_train_OLS = sm.add_constant(X_train)
X_test_OLS = sm.add_constant(X_test)

In [ ]:
X_transformed_df

In [ ]:
X_train_OLS

In [ ]:
feature_names = preprocessor.get_feature_names_out()
feature_names

In [ ]:
# Fit the model using statsmodels
model = sm.OLS(y_train, X_train_OLS).fit()

# Predictions
y_train_pred = model.predict(X_train_OLS)
y_test_pred = model.predict(X_test_OLS)

# Evaluation metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Linear Regression Train MSE: {train_mse}")
print(f"Linear Regression Test MSE: {test_mse}")
print(f"Linear Regression Train RMSE: {train_rmse}")
print(f"Linear Regression Test RMSE: {test_rmse}")
print(f"Linear Regression Train NRMSE (mean): {train_nrmse_mean}")
print(f"Linear Regression Test NRMSE (mean): {test_nrmse_mean}")
print(f"Linear Regression Train NRMSE (range): {train_nrmse_range}")
print(f"Linear Regression Test NRMSE (range): {test_nrmse_range}")

train_r2 = model.rsquared
test_r2 = r2_score(y_test, y_test_pred)
print(f"Linear Regression Train R²: {train_r2}")
print(f"Linear Regression Test R²: {test_r2}")

# Summary of the model
print(model.summary())

In [ ]:
import matplotlib.pyplot as plt

# Plotting
plt.figure(figsize=(14, 6))

# Train set plot
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Train Set: Actual vs Predicted')

# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Assuming X_train has been transformed using the preprocessor and y_train is ready
ridge_model = Ridge(alpha=0.3)
ridge_model.fit(X_train, y_train)

# Predict on the train and test sets
y_train_pred = ridge_model.predict(X_train)
y_test_pred = ridge_model.predict(X_test)

# Calculate MSE and RMSE
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)

# Normalize RMSE
mean_norm = np.mean(y_test)
range_norm = np.ptp(y_test)  # Peak to peak (max - min)

nrmse_mean = rmse / mean_norm
nrmse_range = rmse / range_norm

print(f"Mean Squared Error: {mse}")
print(f"RMSE: {rmse}")
print(f"NRMSE (mean): {nrmse_mean}")
print(f"NRMSE (range): {nrmse_range}")

import pandas as pd

# Get the feature names from the preprocessor
feature_names = preprocessor.get_feature_names_out()

# Get coefficients excluding the intercept (if intercept is included)

# Reshape the coefficients if they are incorrectly shaped
if ridge_model.coef_.ndim > 1:
    coef_ = ridge_model.coef_.flatten()  # This converts it from (n_features, 1) to (n_features,)
else:
    coef_ = ridge_model.coef_

# Create DataFrame
coefficients = pd.DataFrame({
    'Coefficient': coef_
}, index=feature_names)


# Print the coefficients
print(coefficients)

# Additionally, print the intercept
print(f"Intercept: {ridge_model.intercept_}")

In [ ]:
# Plotting
plt.figure(figsize=(14, 6))

# Train set plot
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Train Set: Actual vs Predicted')

# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import numpy as np

# Initialize the scaler
scaler = StandardScaler()

# Scale the features for SVR
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the SVR model
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)  # You can tune the parameters
svr_model.fit(X_train_scaled, y_train)

# Predict on the train and test sets
y_train_pred = svr_model.predict(X_train_scaled)
y_test_pred = svr_model.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)

# Calculate normalized RMSE (NRMSE)
mean_norm = np.mean(y_test)
range_norm = np.ptp(y_test)  # Peak to peak (max - min)

nrmse_mean = rmse / mean_norm
nrmse_range = rmse / range_norm

print(f"Mean Squared Error: {mse}")
print(f"RMSE: {rmse}")
print(f"NRMSE (mean): {nrmse_mean}")
print(f"NRMSE (range): {nrmse_range}")

# Print the number of support vectors used in the model
print(f"Number of support vectors: {svr_model.n_support_.sum()}")


In [ ]:
# Plotting
plt.figure(figsize=(14, 6))

# Train set plot
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Train Set: Actual vs Predicted')

# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Define a range of alpha values
alpha_range = np.logspace(-4, 4, 50)

# Perform LassoCV to find the best alpha
lasso_cv = LassoCV(alphas=alpha_range, cv=5, random_state=42)
lasso_cv.fit(X_train, y_train)

# Get the best alpha value
best_alpha = lasso_cv.alpha_
print(f"Best alpha: {best_alpha}")

# Make predictions on the training and test data using the best alpha
y_train_pred = lasso_cv.predict(X_train)
y_test_pred = lasso_cv.predict(X_test)

# Evaluate the model
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Lasso Train MSE: {train_mse}")
print(f"Lasso Test MSE: {test_mse}")
print(f"Lasso Train RMSE: {train_rmse}")
print(f"Lasso Test RMSE: {test_rmse}")
print(f"Lasso Train NRMSE (mean): {train_nrmse_mean}")
print(f"Lasso Test NRMSE (mean): {test_nrmse_mean}")
print(f"Lasso Train NRMSE (range): {train_nrmse_range}")
print(f"Lasso Test NRMSE (range): {test_nrmse_range}")

# Display coefficients
coefficients = pd.DataFrame(lasso_cv.coef_, features, columns=['Coefficient'])
print(coefficients)

# Plotting Predicted vs Actual values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, color='blue', alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2)  # Line of equality
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predicted vs. Actual Values')
plt.show()

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error
import numpy as np


# Define a range of alpha and l1_ratio values
alphas = np.logspace(-4, 4, 50)
l1_ratios = np.linspace(0.1, 1.0, 10)

# Perform ElasticNetCV to find the best alpha and l1_ratio
elastic_net_cv = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, random_state=42)
elastic_net_cv.fit(X_train, y_train)

# Get the best alpha and l1_ratio values
best_alpha = elastic_net_cv.alpha_
best_l1_ratio = elastic_net_cv.l1_ratio_
print(f"Best alpha: {best_alpha}")
print(f"Best l1_ratio: {best_l1_ratio}")

# Make predictions on the training and test data using the best alpha and l1_ratio
y_train_pred = elastic_net_cv.predict(X_train)
y_test_pred = elastic_net_cv.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Elastic Net Train MSE: {train_mse}")
print(f"Elastic Net Test MSE: {test_mse}")
print(f"Elastic Net Train RMSE: {train_rmse}")
print(f"Elastic Net Test RMSE: {test_rmse}")
print(f"Elastic Net Train NRMSE (mean): {train_nrmse_mean}")
print(f"Elastic Net Test NRMSE (mean): {test_nrmse_mean}")
print(f"Elastic Net Train NRMSE (range): {train_nrmse_range}")
print(f"Elastic Net Test NRMSE (range): {test_nrmse_range}")

# Display coefficients
coefficients = pd.DataFrame(elastic_net_cv.coef_, features, columns=['Coefficient'])
print(coefficients)

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np


# Polynomial features transformation
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Scale the features
scaler = StandardScaler()
X_train_poly_scaled = scaler.fit_transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

# Train the Polynomial Regression model
poly_model = LinearRegression()
poly_model.fit(X_train_poly_scaled, y_train)

# Predict
y_train_pred = poly_model.predict(X_train_poly_scaled)
y_test_pred = poly_model.predict(X_test_poly_scaled)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Polynomial Train MSE: {train_mse}")
print(f"Polynomial Test MSE: {test_mse}")
print(f"Polynomial Train RMSE: {train_rmse}")
print(f"Polynomial Test RMSE: {test_rmse}")
print(f"Polynomial Train NRMSE (mean): {train_nrmse_mean}")
print(f"Polynomial Test NRMSE (mean): {test_nrmse_mean}")
print(f"Polynomial Train NRMSE (range): {train_nrmse_range}")
print(f"Polynomial Test NRMSE (range): {test_nrmse_range}")

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the k-NN model
knn_model = KNeighborsRegressor(n_neighbors=5)  # You can tune the number of neighbors
knn_model.fit(X_train_scaled, y_train)

# Predict
y_train_pred = knn_model.predict(X_train_scaled)
y_test_pred = knn_model.predict(X_test_scaled)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"k-NN Train MSE: {train_mse}")
print(f"k-NN Test MSE: {test_mse}")
print(f"k-NN Train RMSE: {train_rmse}")
print(f"k-NN Test RMSE: {test_rmse}")
print(f"k-NN Train NRMSE (mean): {train_nrmse_mean}")
print(f"k-NN Test NRMSE (mean): {test_nrmse_mean}")
print(f"k-NN Train NRMSE (range): {train_nrmse_range}")
print(f"k-NN Test NRMSE (range): {test_nrmse_range}")

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train the Decision Tree model
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

# Predict
y_train_pred = tree_model.predict(X_train)
y_test_pred = tree_model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Decision Tree Train MSE: {train_mse}")
print(f"Decision Tree Test MSE: {test_mse}")
print(f"Decision Tree Train RMSE: {train_rmse}")
print(f"Decision Tree Test RMSE: {test_rmse}")
print(f"Decision Tree Train NRMSE (mean): {train_nrmse_mean}")
print(f"Decision Tree Test NRMSE (mean): {test_nrmse_mean}")
print(f"Decision Tree Train NRMSE (range): {train_nrmse_range}")
print(f"Decision Tree Test NRMSE (range): {test_nrmse_range}")

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Random Forest Train MSE: {train_mse}")
print(f"Random Forest Test MSE: {test_mse}")
print(f"Random Forest Train RMSE: {train_rmse}")
print(f"Random Forest Test RMSE: {test_rmse}")
print(f"Random Forest Train NRMSE (mean): {train_nrmse_mean}")
print(f"Random Forest Test NRMSE (mean): {test_nrmse_mean}")
print(f"Random Forest Train NRMSE (range): {train_nrmse_range}")
print(f"Random Forest Test NRMSE (range): {test_nrmse_range}")


In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Train the PLSR model
plsr_model = PLSRegression(n_components=2)  # You can tune the number of components
plsr_model.fit(X_train, y_train)

# Predict
y_train_pred = plsr_model.predict(X_train)
y_test_pred = plsr_model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"PLSR Train MSE: {train_mse}")
print(f"PLSR Test MSE: {test_mse}")
print(f"PLSR Train RMSE: {train_rmse}")
print(f"PLSR Test RMSE: {test_rmse}")
print(f"PLSR Train NRMSE (mean): {train_nrmse_mean}")
print(f"PLSR Test NRMSE (mean): {test_nrmse_mean}")
print(f"PLSR Train NRMSE (range): {train_nrmse_range}")
print(f"PLSR Test NRMSE (range): {test_nrmse_range}")

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

# Adding lagged features

In [ ]:
loaded_alluvial_df

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Assuming 'loaded_alluvial_df' is your DataFrame
# Feature selection
features = ['stream_order_categorical', 'terrain_cluster', 'calculated_well_depth', 'latitude', 'longitude',
            'season_numeric', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
            'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
            'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
            'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11']
target = 'GT_change_in_well_depth'

# Create feature and target datasets
X = loaded_alluvial_df[features]
y = loaded_alluvial_df[target]

# Split based on 'id_micro_watershed'
unique_ids = loaded_alluvial_df['id_micro_watershed'].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

# Create masks for train and test
train_mask = loaded_alluvial_df['id_micro_watershed'].isin(train_ids)
test_mask = loaded_alluvial_df['id_micro_watershed'].isin(test_ids)

# Apply masks
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

In [ ]:
# Preprocessing pipeline setup
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Setting up the ColumnTransformer with OneHotEncoder dropping one category
preprocessor = ColumnTransformer(transformers=[
    ('cat_onehot', OneHotEncoder(drop='first'), ['terrain_cluster']),
    ('cat_ordinal', OrdinalEncoder(), ['stream_order_categorical', 'season_numeric']),
    ('num', StandardScaler(), [name for name in features if 'percentage_area_stream_order_' in name])
], remainder='passthrough')

# Apply transformations and add constant for OLS
from sklearn.pipeline import Pipeline
pipeline_ols = Pipeline([
    ('preprocessor', preprocessor),
    ('add_constant', FunctionTransformer(lambda x: sm.add_constant(x, has_constant='add')))
])

X_train_OLS = pipeline_ols.fit_transform(X_train)
X_test_OLS = pipeline_ols.transform(X_test)

# Fit the OLS model
import statsmodels.api as sm
ols_model = sm.OLS(y_train, X_train_OLS).fit()
print(ols_model.summary())

# Predict using the fitted OLS model
y_train_ols_pred = ols_model.predict(X_train_OLS)
y_test_ols_pred = ols_model.predict(X_test_OLS)

# Check predictions and model evaluation
from sklearn.metrics import mean_squared_error
train_mse = mean_squared_error(y_train, y_train_ols_pred)
test_mse = mean_squared_error(y_test, y_test_ols_pred)
print(f'OLS Train MSE: {train_mse}, Test MSE: {test_mse}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import TransformerMixin
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

# Define the preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat_onehot', OneHotEncoder(drop='first'), ['terrain_cluster']),
    ('cat_ordinal', OrdinalEncoder(), ['stream_order_categorical', 'season_numeric']),
    ('num', StandardScaler(), [name for name in features if 'percentage_area_stream_order_' in name])
], remainder='passthrough')

# Custom transformer to add a constant and handle inf/nan
class AddConstant(TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Convert to DataFrame, add constant, handle infinite values or NaNs
        X_df = pd.DataFrame(X, columns=preprocessor.get_feature_names_out())
        X_df.replace([np.inf, -np.inf], np.nan, inplace=True)  # Replace infinities with nan
        X_df.dropna(inplace=True)  # Drop rows with NaNs
        return sm.add_constant(X_df)  # Add constant

pipeline_ols = make_pipeline(preprocessor, AddConstant())

# Apply the transformations
X_train_OLS = pipeline_ols.fit_transform(X_train)
X_test_OLS = pipeline_ols.transform(X_test)

# Ensure there are no NaNs or infinities
if np.isnan(X_train_OLS).any().any():
    raise ValueError("NaNs present in X_train_OLS")
if np.isinf(X_train_OLS).any().any():
    raise ValueError("Infs present in X_train_OLS")

# Fit the OLS model using statsmodels
ols_model = sm.OLS(y_train, X_train_OLS).fit()
print(ols_model.summary())

# Predict using the fitted OLS model
y_train_ols_pred = ols_model.predict(X_train_OLS)
y_test_ols_pred = ols_model.predict(X_test_OLS)

# Calculate and print MSE for train and test sets
train_mse = mean_squared_error(y_train, y_train_ols_pred)
test_mse = mean_squared_error(y_test, y_test_ols_pred)
print(f'OLS Train MSE: {train_mse}, Test MSE: {test_mse}')

In [ ]:
X

In [ ]:
y

In [ ]:
# Create a unique list of micro-watershed IDs
micro_watershed_ids = latest_loaded_alluvial_df['id_micro_watershed'].unique()

# Split micro-watershed IDs into train and test sets
train_ids, test_ids = train_test_split(micro_watershed_ids, test_size=0.2, random_state=42)

# Assign records to the train or test set based on the micro-watershed ID
train_mask = latest_loaded_alluvial_df['id_micro_watershed'].isin(train_ids)
test_mask = latest_loaded_alluvial_df['id_micro_watershed'].isin(test_ids)

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

# Optional: Reset index if necessary
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

# Preprocessing
categorical_features_onehot = ['terrain_cluster']
categorical_features_ordinal = ['stream_order_categorical', 'season_numeric']
numerical_features = [
    'calculated_well_depth', 'latitude', 'longitude', 'percentage_area_stream_order_1', 'percentage_area_stream_order_2',
    'percentage_area_stream_order_3', 'percentage_area_stream_order_4', 'percentage_area_stream_order_5',
    'percentage_area_stream_order_6', 'percentage_area_stream_order_7', 'percentage_area_stream_order_8',
    'percentage_area_stream_order_9', 'percentage_area_stream_order_10', 'percentage_area_stream_order_11'
]

# Define the column transformer without adding a constant
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_onehot', OneHotEncoder(handle_unknown='ignore'), categorical_features_onehot),
        ('cat_ordinal', OrdinalEncoder(), categorical_features_ordinal),
        ('num', StandardScaler(), numerical_features)
    ])

In [ ]:
# Check the columns in X_train to ensure they include 'terrain_cluster'
print(X_train.columns)

# Also, inspect the first few rows to make sure 'terrain_cluster' contains valid data
print(X_train[['terrain_cluster']].head())

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

# Apply the preprocessor within a pipeline setup
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit the model on the training data
pipeline.fit(X_train, y_train)

In [ ]:
# Predict on the training and testing data
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Evaluate the model
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Linear Regression MSE: Train = {train_mse}, Test = {test_mse}")
print(f"Linear Regression R²: Train = {train_r2}, Test = {test_r2}")

# For usage with statsmodels, add a constant to the transformed datasets
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)
X_test_transformed = pipeline.named_steps['preprocessor'].transform(X_test)

# Adding constants to X_train and X_test for OLS regression
X_train_OLS = sm.add_constant(X_train)  # This will add the constant to X_train
X_test_OLS = sm.add_constant(X_test)  # This will add the constant to X_test

# Check if columns are aligned after adding the constant
print("X_train_OLS columns:", X_train_OLS.shape[1])
print("X_test_OLS columns:", X_test_OLS.shape[1])

# Fit the model using statsmodels
ols_model = sm.OLS(y_train, X_train_OLS).fit()

# Print the summary of the model
print(ols_model.summary())

# Predict using the fitted model
y_train_ols_pred = ols_model.predict(X_train_OLS)
y_test_ols_pred = ols_model.predict(X_test_OLS)

# Evaluate the OLS model
ols_train_mse = mean_squared_error(y_train, y_train_ols_pred)
ols_test_mse = mean_squared_error(y_test, y_test_ols_pred)
ols_train_r2 = r2_score(y_train, y_train_ols_pred)
ols_test_r2 = r2_score(y_test, y_test_ols_pred)

print(f"OLS Regression MSE: Train = {ols_train_mse}, Test = {ols_test_mse}")
print(f"OLS Regression R²: Train = {ols_train_r2}, Test = {ols_test_r2}")

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_train_OLS

In [ ]:
X_test_OLS

In [ ]:
# Fit the model using statsmodels
model = sm.OLS(y_train, X_train_OLS).fit()

# Predictions
y_train_pred = model.predict(X_train_OLS)
y_test_pred = model.predict(X_test_OLS)

# Evaluation metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Linear Regression Train MSE: {train_mse}")
print(f"Linear Regression Test MSE: {test_mse}")
print(f"Linear Regression Train RMSE: {train_rmse}")
print(f"Linear Regression Test RMSE: {test_rmse}")
print(f"Linear Regression Train NRMSE (mean): {train_nrmse_mean}")
print(f"Linear Regression Test NRMSE (mean): {test_nrmse_mean}")
print(f"Linear Regression Train NRMSE (range): {train_nrmse_range}")
print(f"Linear Regression Test NRMSE (range): {test_nrmse_range}")

train_r2 = model.rsquared
test_r2 = r2_score(y_test, y_test_pred)
print(f"Linear Regression Train R²: {train_r2}")
print(f"Linear Regression Test R²: {test_r2}")

# Summary of the model
print(model.summary())

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
latest_loaded_alluvial_df

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Define features and target
features = [
    'stream_order_categorical', 'terrain_cluster', 'calculated_well_depth', 'latitude', 'longitude', 'season_numeric',
    'percentage_area_stream_order_1', 'percentage_area_stream_order_2', 'percentage_area_stream_order_3',
    'percentage_area_stream_order_4', 'percentage_area_stream_order_5', 'percentage_area_stream_order_6',
    'percentage_area_stream_order_7', 'percentage_area_stream_order_8', 'percentage_area_stream_order_9',
    'percentage_area_stream_order_10', 'percentage_area_stream_order_11'
]


target = 'GT_change_in_well_depth'


latest_loaded_alluvial_df=loaded_alluvial_df.copy()
# Adding lagged features (example for previous 3 seasons' well depth changes)
lags = 3

for lag in range(1, lags + 1):
    latest_loaded_alluvial_df[f'lag_{lag}_change_in_well_depth'] = loaded_alluvial_df.groupby('id_micro_watershed')['GT_change_in_well_depth'].shift(lag)

# Drop NA values generated by lagging
latest_loaded_alluvial_df = latest_loaded_alluvial_df.dropna()

# Define features and target again after adding lagged features
lagged_features = [f'lag_{lag}_change_in_well_depth' for lag in range(1, lags + 1)]
all_features = features + lagged_features

X = latest_loaded_alluvial_df[all_features]
y = latest_loaded_alluvial_df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
categorical_features = ['stream_order', 'terrain_cluster', 'season_numeric']
numerical_features = ['calculated_well_depth', 'latitude', 'longitude'] + lagged_features

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

In [ ]:
# Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

# Predict
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Train MSE: {train_mse}")
print(f"Test MSE: {test_mse}")
print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")
print(f"Train NRMSE (mean): {train_nrmse_mean}")
print(f"Test NRMSE (mean): {test_nrmse_mean}")
print(f"Train NRMSE (range): {train_nrmse_range}")
print(f"Test NRMSE (range): {test_nrmse_range}")


In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Test set plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')

plt.tight_layout()
plt.show()

# Grid Search

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from tqdm import tqdm

# Assuming new_loaded_alluvial_df is your DataFrame

# Define features and target
features = ['stream_order', 'terrain_cluster', 'calculated_well_depth', 'latitude', 'longitude', 'season_numeric']
target = 'GT_change_in_well_depth'

X = new_loaded_alluvial_df[features]
y = new_loaded_alluvial_df[target]

# Split data
unique_ids = new_loaded_alluvial_df['id_micro_watershed'].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
train_df = new_loaded_alluvial_df[new_loaded_alluvial_df['id_micro_watershed'].isin(train_ids)]
test_df = new_loaded_alluvial_df[new_loaded_alluvial_df['id_micro_watershed'].isin(test_ids)]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize lists to store results
best_params = None
best_score = float('inf')
all_results = []

# Create the parameter grid
param_list = list(ParameterGrid(param_grid))

# Manual grid search with progress bar
for params in tqdm(param_list, desc="Grid Search Progress"):
    rf = RandomForestRegressor(**params, random_state=42)
    cv_scores = []

    for train_idx, val_idx in cv.split(X_train):
        X_train_cv, X_val_cv = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_cv, y_val_cv = y_train.iloc[train_idx], y_train.iloc[val_idx]

        rf.fit(X_train_cv, y_train_cv)
        y_val_pred = rf.predict(X_val_cv)
        mse = mean_squared_error(y_val_cv, y_val_pred)
        cv_scores.append(mse)

    mean_cv_score = np.mean(cv_scores)
    all_results.append((params, mean_cv_score))

    if mean_cv_score < best_score:
        best_score = mean_cv_score
        best_params = params

print(f"Best parameters: {best_params}")

# Train the best model
best_rf_model = RandomForestRegressor(**best_params, random_state=42)
best_rf_model.fit(X_train, y_train)

# Predict
y_train_pred = best_rf_model.predict(X_train)
y_test_pred = best_rf_model.predict(X_test)

# Evaluate
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_nrmse_mean = train_rmse / np.mean(y_train)
test_nrmse_mean = test_rmse / np.mean(y_test)
train_nrmse_range = train_rmse / np.ptp(y_train)
test_nrmse_range = test_rmse / np.ptp(y_test)

print(f"Random Forest Train MSE: {train_mse}")
print(f"Random Forest Test MSE: {test_mse}")
print(f"Random Forest Train RMSE: {train_rmse}")
print(f"Random Forest Test RMSE: {test_rmse}")
print(f"Random Forest Train NRMSE (mean): {train_nrmse_mean}")
print(f"Random Forest Test NRMSE (mean): {test_nrmse_mean}")
print(f"Random Forest Train NRMSE (range): {train_nrmse_range}")
print(f"Random Forest Test NRMSE (range): {test_nrmse_range}")